# MoA from L1000: which ideas work, tested with seeds and cross-validation (CPU only)

**Task.** Predict the mechanism class (15 classes) of a compound from its L1000 signature in A549 cells: 978 genes × 6 doses, 87 labelled compounds, 29 test compounds, 1,286 unlabelled compounds.

**How to read this notebook.** Every idea is a method `fn(train compounds, compounds to predict) → class probabilities` and goes through the same harness:

- out-of-fold predictions on the official folds **and** on `N_SEEDS` seeded stratified 4-fold splits (by compound);
- each method is compared with a like-for-like reference (same classifier without the idea);
- `diff`, `lo`, `hi` = difference of the seed-averaged macro-F1 and its 95% interval from a bootstrap over compounds; `verdict` is *better* / *worse* only if the interval excludes 0, otherwise *no evidence*;
- prior-knowledge ideas (pathways, STRING, gene pairs) are run next to matched random controls.

With 87 compounds an interval of about ±0.08 is normal, so most ideas cannot be told apart from the baseline; the notebook is built to show that honestly instead of ranking by the third decimal. The last section collects everything and corrects for the number of comparisons.

**Leakage control.** Set `PERMUTE_LABELS = True` in the config cell and run all: the training labels are shuffled and every method must fall to chance (macro-F1 about 0.03 to 0.10). Any method that does not is leaking labels.

**Sections.** Baselines · 1 Dose merging · 2 Batch / plate effects · 3 Pathway gene sets and footprints · 4 Gene graphs (STRING, co-expression, long-range) · 5 Gene pairs and gene order · 6 Easy vs hard classes (re-weighting, two-stage) · 7 Unlabelled compounds (representation, compound graph, self-training) · 8 Other classifiers, ensembling, honest selection · Summary · Candidate model.

numpy / pandas / scipy / scikit-learn / matplotlib only. Run from `src/`; all tables and figures are saved to `results/pathway_graph/`.

## Background: how others approached this (quick literature check)
Numbers below come from a quick read of the cited sources; re-check them before quoting in a presentation.

**The data source.** Way et al. 2022 (*Cell Systems* 13:911, L1000 and Cell Painting in A549):
- L1000 z-scores are taken against all other wells of the same plate; whitening against controls ("spherizing") did not help L1000.
- Signal rises with dose: 16% of compounds have reproducible signatures at the lowest dose, 35% at the highest.
- Compounds sharing a mechanism match in 7–21% of mechanisms within one dose and in 50% when doses are combined.
- Deep models adapted from the Kaggle MoA winners gave "relatively low" performance overall; HSP, MEK, mTOR, EGFR and proteasome inhibitors were among the well-predicted classes.

**Models.** In the Kaggle MoA competition (2020; about 23,800 samples) seven winning architectures differed by about 1% in log-loss, and the second-place blend gained 0.00002 over its best single model (solution READMEs). Model family matters little; signal strength, dose handling and the mechanism itself matter more.

**Similarity measures.** Among six L1000 connectivity scores a signed-rank score was generally best (Lin et al. 2020, *Brief Bioinform* 21:2194). Nearest shrunken centroids are the classical few-samples upgrade of the centroid (Tibshirani et al. 2002, *PNAS* 99:6567).

**Signature quality.** The Connectivity Map flags inactive signatures with a transcriptional activity score (signature strength × replicate correlation); 92% of established drugs but 15% of unoptimised compounds pass it (Subramanian et al. 2017, *Cell* 171:1437). Dose-response modelling of L1000 found plate effects to be small next to compound effects (Wang & Novick 2023, *Bioinformatics*, DOSE-L1000).

**Gene-level priors.** The 978 landmark genes were chosen as low-redundancy cluster centroids (Subramanian et al. 2017). Random graphs captured gene dependencies almost as well as curated ones (Bertin et al. 2019, arXiv:1905.02295; Hashir et al. 2019, arXiv:1910.09600); network and pathway composite features did not beat single genes across six cohorts (Staiger et al. 2012, *PLoS ONE* 7:e34796); importance scores of pathway-structured networks largely reflect topology (Esser-Skala & Fortelny 2023, *npj Syst Biol Appl*). Footprint signatures such as PROGENy have the best evidence, for recovering which pathway was perturbed (Schubert et al. 2018, *Nat Commun* 9:20).

**Hard classes and unlabelled data.** Flat versus hierarchical classification: neither is uniformly better (Babbar et al. 2013, NeurIPS). Sample re-weighting effects fade in over-parameterised models (Byrd & Lipton 2019, ICML). Semi-supervised learning can degrade when the unlabelled pool contains other classes (Oliver et al. 2018, NeurIPS).

**Evaluation.** Selecting among many models on the same data produces gains as large as real differences between algorithms (Cawley & Talbot 2010, *JMLR* 11:2079); cross-validation error bars are about ±10% at 100 samples and the spread across folds understates them (Varoquaux 2017, arXiv:1706.07581). This is why every comparison here is paired, bootstrapped over compounds, run on several seeds, and corrected for the number of comparisons.

## Configuration

In [ ]:
import os, time, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from scipy.optimize import linear_sum_assignment
warnings.filterwarnings("ignore")

ROOT = Path(os.getenv("ROOT", "/data/cureatime" if Path("/data/cureatime").exists() else ".."))   # data (read-only mount in the container)
RES = Path(os.getenv("RES", "../pathway_resources"))                                               # gene-level prior knowledge (this repo)
OUT = Path(os.getenv("OUT_DIR", "../results/pathway_graph")); OUT.mkdir(parents=True, exist_ok=True)
SEED, N_SEEDS, N_BOOT = 2026, int(os.getenv("N_SEEDS", 10)), 1000     # N_SEEDS repeated stratified 4-fold splits (by compound) on top of the official folds
DOSES, HI = [0.04, 0.12, 0.37, 1.11, 3.33, 10.0], [4, 5]              # HI = the two top doses (3.33 and 10 uM)
SHIFTS, GEN_TEMP, CEN_TEMP = [-2, -1, 0, 1, 2], 20.0, 10.0            # reverse expert: potency shifts (dose steps); softmax temperatures
PERMUTE_LABELS = os.getenv("PERMUTE_LABELS", "0") == "1"              # leakage control: True shuffles the training labels, then EVERY method must fall to chance
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 80)

## Data: all signatures, then one tensor per compound

In [ ]:
def _load():
    X = pd.read_csv(ROOT / "X_train.csv.gz", index_col=0); y = pd.read_csv(ROOT / "y_train.csv", index_col=0)["moa"].loc[X.index]
    m = pd.read_csv(ROOT / "meta_train.csv", index_col=0).loc[X.index]
    Xte = pd.read_csv(ROOT / "X_test.csv.gz", index_col=0); mte = pd.read_csv(ROOT / "meta_test.csv", index_col=0).loc[Xte.index]
    genes = list(X.columns); parts = [(X, m.assign(src="train", moa=y)), (Xte[genes], mte.assign(src="test"))]
    if (ROOT / "X_unlabeled.csv.gz").exists():
        Un = pd.read_csv(ROOT / "X_unlabeled.csv.gz", index_col=0); parts.append((Un[genes], Un[["compound", "dose_um"]].assign(src="unl")))
    S = np.concatenate([p[0].values for p in parts]).astype(np.float32); SM = pd.concat([p[1] for p in parts])
    SM["plate"] = SM.index.str.split(":").str[0]; SM["well"] = SM.index.str.split(":").str[1]
    return S, SM, genes

S, SM, genes = _load()
G = len(genes)
_order = SM.drop_duplicates("compound").assign(o=lambda d: d.src.map({"train": 0, "test": 1, "unl": 2})).sort_values(["o", "compound"], kind="stable")
cpds = _order.compound.tolist(); N = len(cpds); n_tr, n_te = (_order.src == "train").sum(), (_order.src == "test").sum(); n_lab = n_tr + n_te
SM["cpd"] = SM.compound.map({c: i for i, c in enumerate(cpds)}).values                               # compound index of every signature
SM["dose_i"] = np.abs(np.log(SM.dose_um.values)[:, None] - np.log(DOSES)[None]).argmin(1)            # nearest of the 6 dose slots
SM["row"] = np.arange(len(SM))

def _slots():
    """for every compound and dose slot, the signature rows to average; a missing dose takes the nearest measured dose"""
    rows = []
    for c, g in SM.groupby("cpd"):
        ld = np.log(g.dose_um.values); u = np.unique(ld)
        for d, dd in enumerate(np.log(DOSES)): rows.append(g.row.values[ld == u[np.abs(u - dd).argmin()]])
    return rows
_SLOT = _slots()

def tensor(S):
    """signatures x genes -> compounds x genes x 6 doses (replicates averaged). One training compound (a proteasome inhibitor, the
    plate positive control) exists only at 20 uM on every plate, so its six slots all hold its mean 20 uM profile."""
    return np.stack([S[r].mean(0) for r in _SLOT]).reshape(N, len(DOSES), -1).transpose(0, 2, 1).astype(np.float32)

A = tensor(S)                                                          # the raw representation: compounds x genes x doses
_moa = SM[SM.src == "train"].groupby("cpd").moa.first(); classes = sorted(_moa.unique()); K = len(classes)
yi = _moa.map({c: i for i, c in enumerate(classes)}).values; yi_all = np.r_[yi, -np.ones(N - n_tr, int)]
fold = SM[SM.src == "train"].groupby("cpd").cv_fold.first().values.astype(int)
if PERMUTE_LABELS:                                                     # shuffled within the official folds, so every fold keeps every class
    _r = np.random.default_rng(SEED)
    for o in np.unique(fold): _i = np.flatnonzero(fold == o); yi[_i] = yi[_r.permutation(_i)]
    yi_all[:n_tr] = yi
TR, TE, UNL = np.arange(n_tr), np.arange(n_tr, n_lab), np.arange(n_lab, N)
cpd_plate = SM.groupby("cpd").plate.agg(lambda s: s.iloc[0] if s.nunique() == 1 else "many").values  # each compound sits on one plate (2 controls: all)
n_deg = (np.abs(A) > 2).sum(1)                                         # compounds x doses: genes with |z| > 2
NOISE = np.median(n_deg[:n_lab, 0])                                    # noise floor: the lowest dose is close to untreated
silent = n_deg[:, HI].mean(1) < 2 * NOISE                              # looks untreated even at the two top doses

## Gene annotation: pathway weights, expected pathways per class, STRING

In [ ]:
PW = pd.read_csv(RES / "l1000_pathway_weights.csv"); _key = "probe" if set(genes) & set(PW.probe) else "symbol"
symbols = PW.drop_duplicates(_key).set_index(_key).symbol.reindex(genes).values                      # gene symbol of every column
PW = PW.drop_duplicates(_key).set_index(_key).drop(columns=["probe", "symbol"], errors="ignore").reindex(genes).fillna(0)   # genes x 52 pathway weights
sets = {p: np.flatnonzero(PW[p].values != 0) for p in PW.columns}                                    # pathway -> gene indices
PRIORS = {"EGFR": "PROGENy:EGFR PROGENy:MAPK", "MEK": "PROGENy:MAPK PROGENy:EGFR", "RAF": "PROGENy:MAPK", "PI3K": "PROGENy:PI3K H:MTORC1_SIGNALING",
          "AKT": "PROGENy:PI3K H:MTORC1_SIGNALING", "mTOR": "H:MTORC1_SIGNALING PROGENy:PI3K", "JAK": "PROGENy:JAK-STAT H:IL6_JAK_STAT3_SIGNALING",
          "CDK": "H:E2F_TARGETS H:G2M_CHECKPOINT", "PLK": "H:G2M_CHECKPOINT H:MITOTIC_SPINDLE", "tubulin": "H:G2M_CHECKPOINT H:MITOTIC_SPINDLE",
          "topoisomerase": "PROGENy:p53 H:E2F_TARGETS", "PARP": "PROGENy:p53 H:DNA_REPAIR", "HSP": "custom:HSF1_heat_shock",
          "proteasome": "H:UNFOLDED_PROTEIN_RESPONSE custom:HSF1_heat_shock"}   # expected pathway readouts per class: the author's mapping from target biology, fixed before modelling (HDAC: none)
PRIORS = {next(c for c in classes if c.startswith(k + " ")): v.split() for k, v in PRIORS.items()}   # keyed by full class name
STRING = pd.read_csv(RES / "l1000_string_edges.csv") if (RES / "l1000_string_edges.csv").exists() else None   # STRING v12 associations among the landmark genes (probe_a, probe_b, score)

## Splits: official folds plus seeded stratified repeats

In [ ]:
def strat_folds(seed, k=4):
    """stratified by class at the compound level: every fold gets at least one compound of every class"""
    rng = np.random.default_rng(seed); f = np.zeros(n_tr, int); j = 0
    for c in rng.permutation(K):
        for i in rng.permutation(np.flatnonzero(yi == c)): f[i] = j % k; j += 1
    return f
SPLITS = {"official": fold, **{f"seed{s}": strat_folds(SEED + s) for s in range(N_SEEDS)}}
SEEDS = [k for k in SPLITS if k != "official"]

## Base classifiers

In [ ]:
def softmax(Z): Z = np.exp(Z - Z.max(1, keepdims=True)); return Z / Z.sum(1, keepdims=True)

def std_pair(Ftr, Fpr):
    """standardize with TRAINING statistics only"""
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6; return (Ftr - mu) / sd, (Fpr - mu) / sd

def corr_rows(H, C):
    H = H - H.mean(1, keepdims=True); C = C - C.mean(1, keepdims=True)
    return (H / (np.linalg.norm(H, axis=1, keepdims=True) + 1e-9)) @ (C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)).T

def clf_centroid(R, tr, pr, y=None, k=None, doses=None):
    """Connectivity-Map idea: compound profile = mean over doses (default: the two top doses); Pearson r to each class centroid"""
    y = yi_all[tr] if y is None else y; k = K if k is None else k
    H = R[:, :, HI if (doses is None and R.shape[2] == len(DOSES)) else (slice(None) if doses is None else doses)].mean(2)
    return softmax(CEN_TEMP * corr_rows(H[pr], np.stack([H[tr][y == c].mean(0) for c in range(k)])))

def clf_lr(R, tr, pr, y=None, k=None, C=0.05, sample_weight=None):
    """regularized multinomial logistic regression on all gene x dose features. With more features than compounds the L2 solution
    lies in the span of the training rows, so the fit is done in that rotated space: identical predictions, far faster."""
    y = yi_all[tr] if y is None else y; k = K if k is None else k
    Ftr, Fpr = std_pair(R[tr].reshape(len(tr), -1), R[pr].reshape(len(pr), -1))
    if Ftr.shape[1] > Ftr.shape[0]: U_, s_, Vt = np.linalg.svd(Ftr, full_matrices=False); Ftr, Fpr = U_ * s_, Fpr @ Vt.T
    m = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(Ftr, y, sample_weight=sample_weight)
    P = np.zeros((len(pr), k)); P[:, m.classes_] = m.predict_proba(Fpr); return P

def clf_reverse(R, tr, pr, y=None, k=None, shifts=SHIFTS, amp=True, with_fit=False):
    """REVERSE (class -> signature): each class is a dose-response template (class mean, gene x dose). A compound is explained by
    every template with its own amplitude (>= 0, if amp) and potency shift along the dose axis; Gaussian log-likelihood with pooled
    within-class variance -> class probabilities. Features are standardized, so amplitude 0 = the average training compound."""
    y = yi_all[tr] if y is None else y; k = K if k is None else k; D = R.shape[2]
    Ftr, Fp = std_pair(R[tr].reshape(len(tr), -1), R[pr].reshape(len(pr), -1)); Ftr3 = Ftr.reshape(len(tr), -1, D)
    T = np.stack([Ftr3[y == c].mean(0) for c in range(k)]); var = np.maximum((Ftr3 - T[y]).reshape(len(tr), -1).var(0), 0.1)
    L = np.full((len(pr), k), -np.inf)
    for c in range(k):
        for s in (shifts if D > 1 else [0]):
            Ts = T[c][:, np.clip(np.arange(D) - s, 0, D - 1)].ravel()
            a = np.clip((Fp / var) @ Ts / ((Ts ** 2 / var).sum() + 1e-12), 0, None) if amp else np.ones(len(Fp))
            L[:, c] = np.maximum(L[:, c], -0.5 * (((Fp - a[:, None] * Ts) ** 2) / var).sum(1))
    P = softmax(L / Fp.shape[1] * GEN_TEMP)
    return (P, (L.max(1) + 0.5 * (Fp ** 2 / var).sum(1)) / Fp.shape[1]) if with_fit else P

CLF = {"centroid": clf_centroid, "LR": clf_lr, "reverse": clf_reverse}

## Evaluation harness

In [ ]:
PRED, INFO = {}, {}                                                    # name -> out-of-fold probabilities (n_splits x n_tr x K); name -> section, reference
Y1 = np.eye(K)[yi]
_BW = np.random.default_rng(SEED).multinomial(n_tr, np.ones(n_tr) / n_tr, N_BOOT).astype(np.float32)   # bootstrap over compounds (as counts)

def oof(fn, f):
    P = np.zeros((n_tr, K))
    for o in np.unique(f): P[f == o] = fn(np.flatnonzero(f != o), np.flatnonzero(f == o))
    return P

def cover(P):
    """every class gets its single best compound; all other compounds keep their argmax"""
    L = np.log(np.clip(P, 1e-6, 1)); n, c = L.shape
    if n <= c: return linear_sum_assignment(-L)[1]
    r, col = linear_sum_assignment(-np.hstack([L, np.repeat(L.max(1, keepdims=True), n - c, 1)]))
    lab = L.argmax(1); lab[r[col < c]] = col[col < c]; return lab

def labels(P, f, cov=False):
    """hard calls from probabilities; with cov, every class is called at least once within each fold"""
    pred = P.argmax(1)
    if cov:
        for o in np.unique(f): pred[f == o] = cover(P[f == o])
    return pred

def macro_f1(pred, Wt=None):
    """pred: (splits x compounds) hard calls. Returns macro-F1 per split, or per (bootstrap, split) if compound weights Wt are given.
    Classes absent from a resample are left out of the average."""
    Wt = np.ones((1, n_tr), np.float32) if Wt is None else Wt; P1 = np.eye(K, dtype=np.float32)[pred]
    tp = np.einsum("bn,snk->bsk", Wt, P1 * Y1[None].astype(np.float32)); pc = np.einsum("bn,snk->bsk", Wt, P1); tc = (Wt @ Y1.astype(np.float32))[:, None]
    f = 2 * tp / np.maximum(pc + tc, 1e-9); ok = tc > 0
    return ((f * ok).sum(-1) / ok.sum(-1)).squeeze(0) if len(Wt) == 1 else (f * ok).sum(-1) / ok.sum(-1)

def calls(name, cov=False): return np.stack([labels(P, f, cov) for P, f in zip(PRED[name], SPLITS.values())])

def evaluate(name, fn=None, section="", ref=None, P=None):
    """run method fn on the official folds and on every seeded split (or register ready-made out-of-fold probabilities P)"""
    t0 = time.time(); PRED[name] = np.stack([oof(fn, f) for f in SPLITS.values()]) if P is None else P
    INFO[name] = dict(section=section, ref=ref, secs=time.time() - t0, fn=fn); return name

def evaluate_rep(rep, R, section="", clfs=("centroid", "LR", "reverse"), ref_rep="raw", **kw):
    """evaluate a representation (compounds x genes x doses) with the three base classifiers and their mean; each is compared with
    the SAME classifier on the reference representation"""
    for c in clfs: evaluate(f"{rep} | {c}", lambda tr, pr, c=c: CLF[c](R, tr, pr, **kw), section, None if rep == ref_rep else f"{ref_rep} | {c}")
    if len(clfs) > 1: evaluate(f"{rep} | ensemble", None, section, None if rep == ref_rep else f"{ref_rep} | ensemble", P=np.mean([PRED[f"{rep} | {c}"] for c in clfs], 0))

def compare(a, b, cov=False):
    """paired comparison of two methods on the seeded splits: difference of the seed-averaged macro-F1, 95% CI from a bootstrap
    over compounds (uncertainty from having only 87 compounds), and in how many seeds a beats b (stability across splits)"""
    pa, pb = calls(a, cov)[1:], calls(b, cov)[1:]; d = (macro_f1(pa, _BW) - macro_f1(pb, _BW)).mean(1); s = macro_f1(pa) - macro_f1(pb)
    return dict(diff=s.mean(), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), seeds_better=f"{(s > 0).sum()}/{len(s)}")

def table(section=None, names=None):
    rows = []
    for n in (names or [n for n in PRED if section is None or INFO[n]["section"] == section]):
        f, fc = macro_f1(calls(n)), macro_f1(calls(n, True)); r = dict(section=INFO[n]["section"], method=n, official=f[0], cv_mean=f[1:].mean(), cv_sd=f[1:].std(), cv_cov=fc[1:].mean(),
                                                                         top3=np.mean([(np.argsort(-P, 1)[:, :3] == yi[:, None]).any(1).mean() for P in PRED[n][1:]]))
        if INFO[n]["ref"]:
            c = compare(n, INFO[n]["ref"]); r.update(vs=INFO[n]["ref"], **c, verdict="better" if c["lo"] > 0 else "worse" if c["hi"] < 0 else "no evidence")
        rows.append(r)
    return pd.DataFrame(rows).round(3)

def save(section, fname):
    T = table(section); T.to_csv(OUT / fname, index=False); return T.drop(columns="section")

def ensemble(name, members, section="", ref=None):
    """mean of the out-of-fold probabilities of already evaluated methods"""
    return evaluate(name, None, section, ref, P=np.mean([PRED[m] for m in members], 0))

def predict_test(fn): return fn(TR, TE)                                # fit on all training compounds, predict the test compounds

## Baselines on the raw data
Three base classifiers and their mean, on the raw tensor. These are the references for everything below.

In [ ]:
evaluate_rep("raw", A, "baseline")
print(f"{n_tr} train, {n_te} test, {N - n_lab} unlabelled compounds | {len(SPLITS)} splits (official + {N_SEEDS} seeds) | labels permuted: {PERMUTE_LABELS}")
save("baseline", "baseline_results.csv")

## 1. Dose: how should the six doses be merged, and does dose structure matter?
Way et al. 2022 report that on this dataset mechanisms match far more often across doses than within one dose. Every merging rule below turns
the 6 doses into one profile per compound (label-free) and is compared with the same classifier on the raw 6-dose tensor. Controls: the dose
order shuffled within each compound (5 draws), which tests whether the order of doses carries information at all.

In [ ]:
dose_act = np.clip(n_deg - NOISE, 0, None).astype(np.float32)                    # activity of each compound x dose above the noise floor

def dose_profile(rule):
    """label-free 6 doses -> 1 profile (compounds x genes x 1)"""
    if rule == "all-dose mean": P = A.mean(2)
    elif rule.startswith("top-"): P = A[:, :, -int(rule[4]):].mean(2)
    elif rule == "activity-weighted mean": w = dose_act + 1e-3; P = (A * w[:, None]).sum(2) / w.sum(1)[:, None]
    elif rule == "MODZ (dose-correlation weights)":
        R = np.argsort(np.argsort(A, 1), 1).astype(np.float32); R -= R.mean(1, keepdims=True); R /= np.linalg.norm(R, axis=1, keepdims=True) + 1e-9
        C = np.einsum("ngd,nge->nde", R, R); w = np.clip((C.sum(2) - 1) / 5, 0, None) + 1e-3   # mean Spearman r of each dose with the other doses
        P = (A * w[:, None]).sum(2) / w.sum(1)[:, None]
    elif rule == "most active dose": P = A[np.arange(N), :, n_deg.argmax(1)]
    elif rule == "signed max |z|": i = np.abs(A).argmax(2); P = np.take_along_axis(A, i[:, :, None], 2)[:, :, 0]
    elif rule == "Stouffer (sum / sqrt 6)": P = A.sum(2) / np.sqrt(6)
    return P[:, :, None].astype(np.float32)

dose_rules = ["all-dose mean", "top-1", "top-2", "top-3", "activity-weighted mean", "MODZ (dose-correlation weights)", "most active dose", "signed max |z|", "Stouffer (sum / sqrt 6)"]
for r in dose_rules: evaluate_rep(f"dose: {r}", dose_profile(r), "dose", clfs=("centroid", "LR"))
for d in range(6): evaluate(f"dose: single dose {DOSES[d]} uM | centroid", lambda tr, pr, d=d: clf_centroid(A[:, :, [d]], tr, pr), "dose", "raw | centroid")

**Dose as instance** (late fusion): every compound-dose is a training row carrying its compound's label; the doses of a compound to predict are
scored separately and pooled. **Dose-response features**: per-gene slope against log-dose, mean, and top-minus-bottom contrast.
**Potency alignment**: each compound's dose axis is shifted so that its activity onset (first dose above 2× the noise floor) sits at 1.11 µM.

In [ ]:
def dose_instances(tr, pr, pool="mean", active_only=False, clf="LR"):
    """rows = (compound, dose); labels only of tr; pooled over the doses of each compound in pr"""
    Xtr = A[tr].transpose(0, 2, 1).reshape(-1, G); ytr = np.repeat(yi_all[tr], 6); keep = np.ones(len(ytr), bool)
    if active_only: keep = (n_deg[tr] > 2 * NOISE).ravel(); keep |= np.repeat(~keep.reshape(-1, 6).any(1), 6)   # silent compounds keep all doses
    Xp = A[pr].transpose(0, 2, 1).reshape(-1, G)
    if clf == "LR":
        Ftr, Fp = std_pair(Xtr[keep], Xp); U_, s_, Vt = np.linalg.svd(Ftr, full_matrices=False)
        m = LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced").fit(U_ * s_, ytr[keep]); Pr = np.zeros((len(Xp), K)); Pr[:, m.classes_] = m.predict_proba(Fp @ Vt.T)
    else: Pr = softmax(CEN_TEMP * corr_rows(Xp, np.stack([Xtr[keep][ytr[keep] == c].mean(0) for c in range(K)])))
    Pr = Pr.reshape(len(pr), 6, K); w = dose_act[pr] + 1e-3
    if pool == "mean": return Pr.mean(1)
    if pool == "activity-weighted": return (Pr * w[:, :, None]).sum(1) / w.sum(1)[:, None]
    M = Pr.max(1); return M / M.sum(1, keepdims=True)

for clf, ref in [("LR", "raw | LR"), ("centroid", "raw | centroid")]:
    for pool in ["mean", "activity-weighted", "max"]: evaluate(f"dose: dose-as-instance, {pool} pooling | {clf}", lambda tr, pr, pool=pool, clf=clf: dose_instances(tr, pr, pool, clf=clf), "dose", ref)
    evaluate(f"dose: dose-as-instance, active doses only, activity-weighted | {clf}", lambda tr, pr, clf=clf: dose_instances(tr, pr, "activity-weighted", True, clf), "dose", ref)

dose_ld = np.log(DOSES) - np.log(DOSES).mean()
dose_DR = np.stack([np.einsum("ngd,d->ng", A, dose_ld) / (dose_ld ** 2).sum(), A.mean(2), A[:, :, 4:].mean(2) - A[:, :, :2].mean(2)], 2).astype(np.float32)   # slope, mean, contrast
evaluate("dose: dose-response features (slope, mean, contrast) | LR", lambda tr, pr: clf_lr(dose_DR, tr, pr), "dose", "raw | LR")
evaluate("dose: slope only | centroid", lambda tr, pr: clf_centroid(dose_DR[:, :, [0]], tr, pr), "dose", "raw | centroid")
dose_onset = np.where((n_deg > 2 * NOISE).any(1), (n_deg > 2 * NOISE).argmax(1), 5)               # silent compounds: onset at the top dose
dose_AL = np.stack([A[i][:, np.clip(np.arange(6) + dose_onset[i] - 3, 0, 5)] for i in range(N)]).astype(np.float32)   # onset moved to slot 3
evaluate_rep("dose: potency-aligned tensor", dose_AL, "dose")

**Reverse-expert ablation** (the class dose-response template, with the compound's own amplitude and potency shift) and the **dose-order control**.

In [ ]:
for nm, kw in [("fixed template (no amplitude, no shift)", dict(shifts=[0], amp=False)), ("amplitude only", dict(shifts=[0])),
               ("amplitude + shift ±1", dict(shifts=[-1, 0, 1])), ("amplitude + shift ±3", dict(shifts=[-3, -2, -1, 0, 1, 2, 3]))]:
    evaluate(f"dose: reverse, {nm} | reverse", lambda tr, pr, kw=kw: clf_reverse(A, tr, pr, **kw), "dose", "raw | reverse")
for j in range(5):
    p = np.random.default_rng(SEED + 100 + j).permuted(np.tile(np.arange(6), (N, 1)), axis=1)          # independent dose order per compound
    dose_P = np.take_along_axis(A, p[:, None, :], 2)
    for c in ["LR", "reverse"]: evaluate(f"dose ctrl: dose order shuffled #{j} | {c}", lambda tr, pr, R=dose_P, c=c: CLF[c](R, tr, pr), "dose_ctrl", f"raw | {c}")
dose_C = table("dose_ctrl"); print("dose order shuffled (5 draws), mean seeded-CV macro-F1:", dose_C.groupby(dose_C.method.str.split("|").str[-1].str.strip()).cv_mean.mean().round(3).to_dict(),
                                  "| raw:", {c: round(table(names=[f"raw | {c}"]).cv_mean[0], 3) for c in ["LR", "reverse"]})

In [ ]:
dose_T = save("dose", "dose_results.csv")
_t = table("dose"); _s = _t[_t.method.str.startswith("dose: single dose")]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(DOSES, _s.cv_mean, "o-"); ax[0].fill_between(DOSES, _s.cv_mean - _s.cv_sd, _s.cv_mean + _s.cv_sd, alpha=.2); ax[0].set_xscale("log")
ax[0].axhline(table(names=["raw | centroid"]).cv_mean[0], ls="--", color="k", lw=.8, label="raw centroid (top-2 mean)"); ax[0].set_xlabel("dose (µM)"); ax[0].set_ylabel("seeded-CV macro-F1")
ax[0].set_title("centroid on one dose", loc="left", fontsize=9); ax[0].legend(fontsize=7, frameon=False)
_d = _t[_t.vs.notna() & ~_t.method.str.startswith("dose: single dose")].sort_values("diff").reset_index(drop=True)
ax[1].hlines(range(len(_d)), _d.lo, _d.hi, color="grey"); ax[1].scatter(_d["diff"], range(len(_d)), c=_d.verdict.map({"better": "g", "worse": "r"}).fillna("k"), s=12, zorder=3)
ax[1].axvline(0, color="k", lw=.6); ax[1].set_yticks(range(len(_d))); ax[1].set_yticklabels(_d.method.str.replace("dose: ", ""), fontsize=6); ax[1].set_xlabel("Δ macro-F1 vs same classifier on raw")
fig.tight_layout(); fig.savefig(OUT / "fig_dose.png", dpi=150); dose_T

## 2. Batch: is there a plate effect, and does correcting it help?
25 plates of about 360 wells; each compound has all its doses on one plate, and two plate controls (20 µM) sit on every plate.
**A** quantifies the plate effect without labels. **B** applies label-free signature-level corrections and compares each with the same classifier on raw;
the control is plate centring with plate labels shuffled among wells (3 draws). **C** asks whether plate membership inflates cross-validation:
leave-plates-out folds against random unstratified compound folds with the same constraint.

In [ ]:
batch_plate = SM.plate.values; batch_low = (SM.dose_i.values == 0) & (SM.dose_um.values < 1)             # 0.04 uM signatures: closest to untreated
batch_rng = np.random.default_rng(SEED + 200); batch_rows = []
def batch_add(metric, value, note=""): batch_rows.append(dict(metric=metric, value=round(float(value), 4), note=note))
for lab, m in [("all wells", np.ones(len(S), bool)), ("0.04 uM wells", batch_low)]:
    med = pd.DataFrame(S[m]).groupby(batch_plate[m]).median(); mad = pd.DataFrame(np.abs(S[m] - med.loc[batch_plate[m]].values)).groupby(batch_plate[m]).median() * 1.4826
    batch_add(f"median |per-plate gene median| ({lab})", np.median(np.abs(med.values)), "≈0 if z-scores are centred on this population")
    batch_add(f"median per-plate gene MAD ({lab})", np.median(mad.values), "≈1 if z-scores are scaled on this population")
def batch_eta2(X, g):
    gm = pd.DataFrame(X).groupby(g).transform("mean").values; return ((gm - X.mean(0)) ** 2).sum(0) / ((X - X.mean(0)) ** 2).sum(0)
e = batch_eta2(S[batch_low], batch_plate[batch_low]); e0 = np.median([np.median(batch_eta2(S[batch_low], batch_rng.permutation(batch_plate[batch_low]))) for _ in range(20)])
batch_add("median variance explained by plate, per gene (0.04 uM wells)", np.median(e), f"plate labels shuffled: {e0:.4f}")
for c in [c for c in SM.compound.unique() if SM[SM.compound == c].plate.nunique() > 1]:
    m = (SM.compound == c).values; X = S[m]; e = batch_eta2(X, batch_plate[m]); e0 = np.median([np.median(batch_eta2(X, batch_rng.permutation(batch_plate[m]))) for _ in range(20)])
    batch_add(f"plate control {c} ({SM[m].src.iloc[0]}): median variance explained by plate", np.median(e), f"shuffled: {e0:.4f}; {m.sum()} wells")
    C = np.corrcoef(X); same = batch_plate[m][:, None] == batch_plate[m][None]; np.fill_diagonal(C, np.nan)
    batch_add(f"plate control {c}: mean r same plate vs other plates", np.nanmean(C[same]) - np.nanmean(C[~same]), f"same {np.nanmean(C[same]):.3f}, other {np.nanmean(C[~same]):.3f}")
def batch_purity(X, g, k=5):
    C = corr_rows(X, X); np.fill_diagonal(C, -9); nn = np.argsort(-C, 1)[:, :k]; return (g[nn] == g[:, None]).mean()
batch_add("kNN plate purity, 0.04 uM signatures (k=5)", batch_purity(S[batch_low], batch_plate[batch_low]), f"shuffled: {batch_purity(S[batch_low], batch_rng.permutation(batch_plate[batch_low])):.4f}; chance ≈ 1/25")
_cp = np.array([p != "many" for p in cpd_plate]); _H = A[:, :, HI].mean(2)
batch_add("kNN plate purity, top-dose compound profiles (k=5)", batch_purity(_H[_cp], cpd_plate[_cp]), f"shuffled: {batch_purity(_H[_cp], batch_rng.permutation(cpd_plate[_cp])):.4f}")
for part, ix in [("row", 0), ("column", 1)]:
    g = SM.well.str[0].values if ix == 0 else SM.well.str[1:].values
    batch_add(f"median variance explained by well {part} (0.04 uM wells)", np.median(batch_eta2(S[batch_low], g[batch_low])), f"shuffled: {np.median(batch_eta2(S[batch_low], batch_rng.permutation(g[batch_low]))):.4f}")
batch_D = pd.DataFrame(batch_rows); batch_D.to_csv(OUT / "batch_diagnostics.csv", index=False); batch_D

### B. Label-free corrections (signature level, then rebuilt into compound tensors)

In [ ]:
def batch_centre(X, groups, ref=None):
    """subtract the per-group per-gene median, computed on the reference wells of the group (default: all wells)"""
    ref = np.ones(len(X), bool) if ref is None else ref; med = pd.DataFrame(X[ref]).groupby(groups[ref]).median()
    return X - med.reindex(groups).fillna(0).values
def batch_scale(X, groups):
    mad = pd.DataFrame(np.abs(X)).groupby(groups).transform("median").values * 1.4826 + 1e-3; return X / mad * np.median(mad)
def batch_remove_pcs(X, k):
    Xl = X[batch_low] - X[batch_low].mean(0); V = np.linalg.svd(Xl, full_matrices=False)[2][:k]; return X - (X - X[batch_low].mean(0)) @ V.T @ V
def batch_whiten(X, full=False):
    if not full: sd = X[batch_low].std(0) + 1e-3; return X / sd * np.median(sd)
    from sklearn.covariance import LedoitWolf
    C = LedoitWolf().fit(X[batch_low]).covariance_; w, V = np.linalg.eigh(C); W = V @ np.diag(w ** -0.5) @ V.T
    return (X - X[batch_low].mean(0)) @ W * np.sqrt(np.median(w))
batch_corr = {"plate centring (all wells)": batch_centre(S, batch_plate), "plate centring (0.04 uM wells)": batch_centre(S, batch_plate, batch_low),
              "plate centring + plate robust scaling": batch_scale(batch_centre(S, batch_plate), batch_plate),
              "remove 1 PC of 0.04 uM wells": batch_remove_pcs(S, 1), "remove 2 PCs": batch_remove_pcs(S, 2), "remove 5 PCs": batch_remove_pcs(S, 5),
              "whitening, diagonal (0.04 uM SD)": batch_whiten(S), "whitening, shrunk full (spherize on 0.04 uM)": batch_whiten(S, True),
              "global per-gene robust scaling": S / (np.median(np.abs(S - np.median(S, 0)), 0) * 1.4826 + 1e-3) * np.median(np.median(np.abs(S - np.median(S, 0)), 0) * 1.4826)}
for nm, X in batch_corr.items(): evaluate_rep(f"batch: {nm}", tensor(X.astype(np.float32)), "batch")
evaluate_rep("batch: subtract the compound's own 0.04 uM profile", A - A[:, :, [0]], "batch")
for j in range(3): evaluate_rep(f"batch ctrl: centring with shuffled plate labels #{j}", tensor(batch_centre(S, np.random.default_rng(SEED + 210 + j).permutation(batch_plate)).astype(np.float32)), "batch_ctrl", clfs=("centroid", "LR"))
del batch_corr

### C. Does plate membership inflate cross-validation?
Leave-plates-out: whole plates go to one fold (5 random groupings, each retried until every class keeps a training compound in every fold).
Reference: random compound-level folds under the same constraint but without stratification (5 draws), so that only the plate grouping differs.

In [ ]:
def batch_folds(groups, seed):
    rng = np.random.default_rng(seed)
    while True:
        u = rng.permutation(np.unique(groups)); f = pd.Series(np.arange(len(u)) % 4, index=u).loc[groups].values
        if all(len(np.unique(yi[f != o])) == K for o in range(4)): return f
batch_cv = []
for kind in ["leave plates out", "random compound folds"]:
    for j in range(5):
        f = batch_folds(cpd_plate[TR] if kind == "leave plates out" else np.arange(n_tr), SEED + 220 + j)
        for c in ["centroid", "LR", "reverse"]: batch_cv.append(dict(folds=kind, draw=j, clf=c, macro_f1=macro_f1(labels(oof(lambda tr, pr, c=c: CLF[c](A, tr, pr), f), f)[None])[0]))
batch_cv = pd.DataFrame(batch_cv); batch_cv.to_csv(OUT / "batch_plate_cv.csv", index=False)
print(batch_cv.pivot_table(index="clf", columns="folds", values="macro_f1", aggfunc=["mean", "std"]).round(3).to_string())
_w = labels(PRED["raw | centroid"][0], fold); _wr = np.flatnonzero(_w != yi); _obs, _exp = [], []
for i in _wr:                                                                    # wrong official-fold calls: does the called class sit on the same plate?
    mates = np.flatnonzero((cpd_plate[TR] == cpd_plate[i]) & (fold != fold[i])); cl = set(yi[mates]) - {yi[i]}
    _obs.append(_w[i] in cl); _exp.append(len(cl) / (K - 1))
print(f"wrong official-fold centroid calls: {len(_wr)}; called class present among same-plate training compounds: {np.mean(_obs):.2f} (expected by chance {np.mean(_exp):.2f})")

In [ ]:
batch_T = save("batch", "batch_results.csv")
_t = table("batch"); _t = _t[_t.method.str.endswith("| ensemble") | _t.method.str.endswith("| centroid")].sort_values("diff").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5)); ax.hlines(range(len(_t)), _t.lo, _t.hi, color="grey"); ax.scatter(_t["diff"], range(len(_t)), c=_t.verdict.map({"better": "g", "worse": "r"}).fillna("k"), s=12, zorder=3)
ax.axvline(0, color="k", lw=.6); ax.set_yticks(range(len(_t))); ax.set_yticklabels(_t.method.str.replace("batch: ", ""), fontsize=6); ax.set_xlabel("Δ macro-F1 vs same classifier on raw")
ax.set_title("Batch corrections (centroid and ensemble rows)", loc="left", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig_batch.png", dpi=150); batch_T

## 3. Gene sets: do curated pathway gene sets and footprint scores help?
Four uses of the 52 pathway readouts (`pathway_resources/`): restricting to pathway genes, pathway-activity (footprint) features, one expert per
pathway merged, and a class-specific check of the a-priori `PRIORS` mapping. Every one is run next to matched random controls: random gene sets
of the same size (and variance-matched ones), pathway weights with gene labels shuffled (same sparsity and signs), and 52 data-driven gene modules.

In [ ]:
gsets_rng = np.random.default_rng(SEED + 300)
gsets_in = np.flatnonzero((PW.values != 0).any(1)); gsets_out = np.setdiff1d(np.arange(G), gsets_in)
evaluate_rep(f"gsets: {len(gsets_in)} genes in any pathway", A[:, gsets_in], "gsets", clfs=("centroid", "reverse"))
evaluate_rep(f"gsets: {len(gsets_out)} genes in no pathway", A[:, gsets_out], "gsets", clfs=("centroid", "reverse"))
for j in range(5):
    for n in [len(gsets_in), len(gsets_out)]: evaluate_rep(f"gsets ctrl: {n} random genes #{j}", A[:, np.sort(gsets_rng.choice(G, n, replace=False))], "gsets_ctrl", clfs=("centroid", "reverse"))
_c = table("gsets_ctrl"); print("random gene subsets, mean seeded-CV macro-F1:", _c.groupby(_c.method.str.extract(r"ctrl: (\d+) random")[0] + _c.method.str.extract(r"\| (\w+)$")[0]).cv_mean.mean().round(3).to_dict())

**Footprint features**: the tensor projected on the pathway weights (52 scores × 6 doses, scaled over all compounds, label-free), alone, fused with
the gene-level model (mean of probabilities) and concatenated. Controls: the same weights with gene labels shuffled (5 draws) and 52 gene modules
from clustering genes on the unlabelled signatures.

In [ ]:
def gsets_project(Wm):
    F = np.einsum("ngd,gp->npd", A, Wm.astype(np.float32)); return ((F - F.mean((0, 2), keepdims=True)) / (F.std((0, 2), keepdims=True) + 1e-6)).astype(np.float32)
gsets_FP = gsets_project(PW.values)
evaluate_rep("gsets: footprint scores (52 x 6)", gsets_FP, "gsets")
for c in ["centroid", "LR", "reverse"]: ensemble(f"gsets: raw + footprint scores, late fusion | {c}", [f"raw | {c}", f"gsets: footprint scores (52 x 6) | {c}"], "gsets", f"raw | {c}")
evaluate("gsets: raw + footprint scores, concatenated | LR", lambda tr, pr: clf_lr(np.concatenate([A, gsets_FP], 1), tr, pr), "gsets", "raw | LR")
_sl = np.log(DOSES) - np.log(DOSES).mean(); gsets_slope = (np.einsum("npd,d->np", gsets_FP, _sl) / (_sl ** 2).sum())[:, :, None]
evaluate_rep("gsets: footprint dose-slopes (52)", gsets_slope, "gsets", clfs=("centroid", "LR"), ref_rep="gsets: footprint scores (52 x 6)")
from sklearn.cluster import KMeans
_U = S[SM.src.values == "unl"]; _Z = ((_U - _U.mean(0)) / (_U.std(0) + 1e-6)).T; _Z /= np.linalg.norm(_Z, axis=1, keepdims=True)
gsets_mod = KMeans(52, n_init=4, random_state=SEED).fit_predict(_Z); gsets_MW = np.eye(52)[gsets_mod] / np.bincount(gsets_mod, minlength=52)
evaluate_rep("gsets: 52 data-driven gene modules (unlabelled), x 6", gsets_project(gsets_MW), "gsets", ref_rep="gsets: footprint scores (52 x 6)")
for j in range(5): evaluate_rep(f"gsets ctrl: pathway weights, gene labels shuffled #{j}", gsets_project(PW.values[gsets_rng.permutation(G)]), "gsets_ctrl", ref_rep="gsets: footprint scores (52 x 6)")
_c = table("gsets_ctrl"); _c = _c[_c.method.str.contains("shuffled")]
print("footprints with gene labels shuffled, mean seeded-CV macro-F1:", _c.groupby(_c.method.str.extract(r"\| (\w+)$")[0]).cv_mean.mean().round(3).to_dict())

**One expert per pathway**: the reverse expert on each pathway's genes, merged by the mean of log-probabilities, or by the 10 best experts chosen
inside each training part by an inner 3-fold CV. The same procedure runs on random gene sets with the pathway sizes (4 draws for the mean, 2 for
top-10) and on variance-matched random sets (each pathway gene replaced by a random gene from the same variance decile).

In [ ]:
_v = A[:, :, HI].mean(2).var(0); gsets_dec = np.digitize(_v, np.quantile(_v, np.linspace(0, 1, 11)[1:-1]))
def gsets_random(seed, matched=False):
    r = np.random.default_rng(seed)
    if not matched: return [np.sort(r.choice(G, len(ix), replace=False)) for ix in sets.values()]
    return [np.unique([r.choice(np.flatnonzero(gsets_dec == gsets_dec[g])) for g in ix]) for ix in sets.values()]
def gsets_merge(tr, pr, idxs, top=None):
    """reverse expert per gene set; mean of log-probabilities over all sets, or over the `top` sets chosen by inner CV on tr only"""
    L = np.stack([np.log(np.clip(clf_reverse(A[:, ix], tr, pr), 1e-6, 1)) for ix in idxs])
    if top:
        inner = np.zeros(len(tr), int); inner[np.argsort(np.random.default_rng(len(tr)).random(len(tr)))] = np.arange(len(tr)) % 3
        sc = np.zeros(len(idxs))
        for i, ix in enumerate(idxs):
            pi = np.zeros(len(tr), int)
            for o in range(3): pi[inner == o] = clf_reverse(A[:, ix], tr[inner != o], tr[inner == o]).argmax(1)
            sc[i] = np.mean([np.mean(pi[yi_all[tr] == c] == c) for c in np.unique(yi_all[tr])])   # inner balanced accuracy
        L = L[np.argsort(-sc)[:top]]
    return softmax(L.mean(0))
gsets_P = list(sets.values())
evaluate("gsets: one reverse expert per pathway, mean | reverse", lambda tr, pr: gsets_merge(tr, pr, gsets_P), "gsets", "raw | reverse")
evaluate("gsets: one reverse expert per pathway, top-10 by inner CV | reverse", lambda tr, pr: gsets_merge(tr, pr, gsets_P, 10), "gsets", "raw | reverse")
for j in range(4): evaluate(f"gsets ctrl: random sets, mean #{j} | reverse", lambda tr, pr, s=gsets_random(SEED + 310 + j): gsets_merge(tr, pr, s), "gsets_ctrl", "gsets: one reverse expert per pathway, mean | reverse")
for j in range(2): evaluate(f"gsets ctrl: variance-matched random sets, mean #{j} | reverse", lambda tr, pr, s=gsets_random(SEED + 320 + j, True): gsets_merge(tr, pr, s), "gsets_ctrl", "gsets: one reverse expert per pathway, mean | reverse")
for j in range(2): evaluate(f"gsets ctrl: random sets, top-10 #{j} | reverse", lambda tr, pr, s=gsets_random(SEED + 330 + j): gsets_merge(tr, pr, s, 10), "gsets_ctrl", "gsets: one reverse expert per pathway, top-10 by inner CV | reverse")
_c = table("gsets_ctrl"); _c = _c[_c.method.str.contains("sets, ")]
print(_c.groupby(_c.method.str.replace(r" #\d+", "", regex=True)).cv_mean.agg(["mean", "min", "max"]).round(3).to_string())
print(table(names=["gsets: one reverse expert per pathway, mean | reverse", "gsets: one reverse expert per pathway, top-10 by inner CV | reverse"])[["method", "cv_mean"]].to_string(index=False))

**Expected pathway vs random genes, per class** (descriptive). AUC of "this class vs the rest" for the reverse expert on the class's expected
pathway (`PRIORS`, fixed a priori), out-of-fold on the official folds, against 100 random gene sets of the same size. Loud classes reach AUC ≈ 1
from any gene set, so only quiet classes can show a pathway-specific signal. With 26 pairs, about 1.3 are expected below p = 0.05 by chance.

In [ ]:
from sklearn.metrics import roc_auc_score
gsets_null, gsets_rows = {}, []
for j, p in enumerate(sorted({p for ps in PRIORS.values() for p in ps})):
    r = np.random.default_rng(SEED + 340 + j); gsets_null[p] = [oof(lambda tr, pr, ix=r.choice(G, len(sets[p]), replace=False): clf_reverse(A[:, ix], tr, pr), fold) for _ in range(100)]
for cls, ps in PRIORS.items():
    c = classes.index(cls)
    for p in ps:
        a = roc_auc_score(yi == c, oof(lambda tr, pr: clf_reverse(A[:, sets[p]], tr, pr), fold)[:, c]); rr = np.array([roc_auc_score(yi == c, P[:, c]) for P in gsets_null[p]])
        gsets_rows.append(dict(moa=cls, pathway=p, n_genes=len(sets[p]), auc=a, random_median=np.median(rr), random_95pct=np.percentile(rr, 95), p_empirical=(1 + (rr >= a).sum()) / (len(rr) + 1), _r=rr))
gsets_BIO = pd.DataFrame(gsets_rows); gsets_BIO.drop(columns="_r").round(3).to_csv(OUT / "gsets_priors_auc.csv", index=False)
print(f"class-pathway pairs with p < 0.05: {(gsets_BIO.p_empirical < 0.05).sum()} of {len(gsets_BIO)} (about {0.05 * len(gsets_BIO):.1f} expected by chance)")
fig, ax = plt.subplots(figsize=(8, 7))
for j, r in gsets_BIO.iterrows():
    ax.scatter(r["_r"], np.full(len(r["_r"]), j) + np.random.default_rng(j).uniform(-.25, .25, len(r["_r"])), s=3, color="grey", alpha=.35)
    ax.scatter(r.auc, j, s=45, color="crimson" if r.p_empirical < 0.05 else "k", marker="D", zorder=3)
ax.set_yticks(range(len(gsets_BIO))); ax.set_yticklabels([f"{m.replace(' inhibitor', '')}  ←  {p}" for m, p in zip(gsets_BIO.moa, gsets_BIO.pathway)], fontsize=7); ax.invert_yaxis()
ax.set_xlabel("AUC, class vs rest (out-of-fold, official folds)"); fig.suptitle("Expected pathway (diamond; red: p < 0.05) vs random gene sets of the same size (grey)", x=0.02, ha="left", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "fig_gsets.png", dpi=150)
gsets_BIO.drop(columns="_r").round(3)

In [ ]:
gsets_T = save("gsets", "gsets_results.csv"); gsets_T

## 4. Gene graphs: do gene-gene relations (STRING, co-expression), including long-range ones, help?
Graphs: STRING v12 associations among the landmark genes at score ≥ 0.4 and ≥ 0.7 (weighted by score), and a co-expression kNN graph from the
unlabelled signatures using **signed** correlation (positive edges only). First a label-free check that the graph carries expression structure,
then operators on the gene axis: lazy 1-hop smoothing, random walk with restart and heat diffusion (long-range), multi-scale features that keep
the original signal, and the high-pass residual. Controls: the same graph rewired with degrees preserved, and with gene labels permuted (3 draws each).

In [ ]:
graph_gi = {g: i for i, g in enumerate(genes)}
def graph_string(th):
    E = STRING[STRING.score >= th]; W = np.zeros((G, G), np.float32); a, b = E.probe_a.map(graph_gi).values, E.probe_b.map(graph_gi).values
    W[a, b] = W[b, a] = E.score.values; return W
_U = S[SM.src.values == "unl"]; graph_C = np.corrcoef(_U.T); np.fill_diagonal(graph_C, 0)
def graph_coexp(k=10):
    W = np.zeros((G, G), np.float32); nn = np.argsort(-graph_C, 1)[:, :k]; W[np.repeat(np.arange(G), k), nn.ravel()] = np.clip(graph_C[np.repeat(np.arange(G), k), nn.ravel()], 0, None)
    return np.maximum(W, W.T)
GRAPHS = {"STRING >= 0.4": graph_string(0.4), "STRING >= 0.7": graph_string(0.7), "co-expression kNN (k=10, r > 0)": graph_coexp()}
def graph_rewire(W, seed, n_swaps=None):
    """degree-preserving rewiring by double edge swaps (edge weights travel with the edges)"""
    r = np.random.default_rng(seed); iu = np.argwhere(np.triu(W) > 0); w = W[iu[:, 0], iu[:, 1]]; E = iu.copy(); S_ = {tuple(e) for e in E}
    for _ in range(n_swaps or 5 * len(E)):
        i, j = r.integers(len(E), size=2); (a, b), (c, d) = E[i], E[j]
        if len({a, b, c, d}) < 4: continue
        n1, n2 = tuple(sorted((a, d))), tuple(sorted((c, b)))
        if n1 in S_ or n2 in S_: continue
        S_ -= {tuple(sorted((a, b))), tuple(sorted((c, d)))}; S_ |= {n1, n2}; E[i], E[j] = n1, n2
    R = np.zeros_like(W); R[E[:, 0], E[:, 1]] = w; return np.maximum(R, R.T)

**Does the graph carry expression structure?** (label-free, descriptive) Correlation of connected genes across the unlabelled signatures, and
single-gene inference: R² of predicting each gene from its graph neighbours (ridge, held-out signatures) against the same number of random genes.

In [ ]:
_ts = np.random.default_rng(SEED + 400).random(len(_U)) < 0.5; _Z = (_U - _U[_ts].mean(0)) / (_U[_ts].std(0) + 1e-6)
def graph_sgi(W, rand=False, seed=0):
    r = np.random.default_rng(seed); out = []
    for g in np.flatnonzero((W > 0).any(1))[::4]:                                   # every 4th connected gene
        nb = np.flatnonzero(W[g] > 0)[:30]
        if rand: nb = r.choice(np.setdiff1d(np.arange(G), [g]), len(nb), replace=False)
        X, y = _Z[:, nb], _Z[:, g]; b = np.linalg.solve(X[_ts].T @ X[_ts] + 10 * np.eye(len(nb)), X[_ts].T @ y[_ts]); out.append(1 - ((y[~_ts] - X[~_ts] @ b) ** 2).mean() / y[~_ts].var())
    return np.median(out)
graph_rows = []
for nm, W in GRAPHS.items():
    iu = np.argwhere(np.triu(W) > 0); rr = np.random.default_rng(SEED + 401).integers(G, size=(len(iu), 2))
    graph_rows.append(dict(graph=nm, edges=len(iu), genes_with_edges=int((W > 0).any(1).sum()), mean_r_edges=graph_C[iu[:, 0], iu[:, 1]].mean(), mean_r_random_pairs=graph_C[rr[:, 0], rr[:, 1]].mean(),
                           sgi_r2_neighbours=graph_sgi(W), sgi_r2_random_genes=graph_sgi(W, True, SEED + 402)))
for lo, hi in [(0.4, 0.7), (0.7, 0.9), (0.9, 1.01)]:
    iu = np.argwhere(np.triu(GRAPHS["STRING >= 0.4"]) > 0); sc = GRAPHS["STRING >= 0.4"][iu[:, 0], iu[:, 1]]; m = (sc >= lo) & (sc < hi)
    graph_rows.append(dict(graph=f"STRING score in [{lo}, {min(hi, 1)})", edges=int(m.sum()), mean_r_edges=graph_C[iu[m, 0], iu[m, 1]].mean()))
graph_D = pd.DataFrame(graph_rows).round(4); graph_D.to_csv(OUT / "graph_diagnostics.csv", index=False); graph_D

**Operators** (label-free, applied to every dose's gene vector; Â = symmetric-normalised adjacency with self-loops, L = I − Â).

In [ ]:
def graph_ops(W):
    Wl = W + np.eye(G, dtype=np.float32); d = Wl.sum(1) ** -0.5; Ah = d[:, None] * Wl * d[None]; Lp = np.eye(G) - Ah; w, V = np.linalg.eigh(Lp)
    return {"lazy 1-hop smoothing (mix 0.5)": 0.5 * np.eye(G) + 0.5 * Ah,
            "random walk with restart 0.2 (long-range)": 0.2 * np.linalg.inv(np.eye(G) - 0.8 * Ah),
            "heat diffusion t=2 (long-range)": (V * np.exp(-2 * w)) @ V.T,
            "high-pass residual X - ÂX": Lp,
            "multi-scale [X, ÂX, Â²X]": [np.eye(G), Ah, Ah @ Ah]}
def graph_apply(M):
    if isinstance(M, list): return np.concatenate([np.einsum("ij,njd->nid", m.astype(np.float32), A) for m in M], 1)
    return np.einsum("ij,njd->nid", M.astype(np.float32), A)
for gn, W in GRAPHS.items():
    for on, M in graph_ops(W).items():
        if gn == "STRING >= 0.7" and on not in ("lazy 1-hop smoothing (mix 0.5)", "random walk with restart 0.2 (long-range)"): continue
        evaluate_rep(f"graph: {gn}, {on}", graph_apply(M), "graph", clfs=("centroid", "LR") if "multi" in on else ("centroid", "LR", "reverse"))
for j in range(3):
    for kind, W in [("rewired (degrees kept)", graph_rewire(GRAPHS["STRING >= 0.4"], SEED + 410 + j)), ("gene labels permuted", (lambda p: GRAPHS["STRING >= 0.4"][p][:, p])(np.random.default_rng(SEED + 420 + j).permutation(G)))]:
        ops = graph_ops(W)
        for on in ["lazy 1-hop smoothing (mix 0.5)", "random walk with restart 0.2 (long-range)"]:
            evaluate_rep(f"graph ctrl: STRING >= 0.4 {kind} #{j}, {on}", graph_apply(ops[on]), "graph_ctrl", clfs=("centroid", "LR", "reverse"), ref_rep=f"graph: STRING >= 0.4, {on}")
_c = table("graph_ctrl"); _c["op"] = _c.method.str.extract(r", (lazy|random walk)")[0]; _c["kind"] = _c.method.str.extract(r"(rewired|permuted)")[0]; _c["clf"] = _c.method.str.extract(r"\| (\w+)$")[0]
_r = table("graph"); _r = _r[_r.method.str.startswith("graph: STRING >= 0.4, lazy") | _r.method.str.startswith("graph: STRING >= 0.4, random walk")]
print("STRING >= 0.4 vs its controls, seeded-CV macro-F1 (controls: mean of 3 draws):")
print(pd.concat([_c.pivot_table(index=["op", "clf"], columns="kind", values="cv_mean"), _r.assign(op=_r.method.str.extract(r", (lazy|random walk)")[0], clf=_r.method.str.extract(r"\| (\w+)$")[0]).set_index(["op", "clf"]).cv_mean.rename("STRING")], axis=1).round(3).to_string())

In [ ]:
graph_T = save("graph", "graph_results.csv")
_t = table("graph"); _t = _t[_t.vs.notna()].sort_values("diff").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 7)); ax.hlines(range(len(_t)), _t.lo, _t.hi, color="grey"); ax.scatter(_t["diff"], range(len(_t)), c=_t.verdict.map({"better": "g", "worse": "r"}).fillna("k"), s=12, zorder=3)
ax.axvline(0, color="k", lw=.6); ax.set_yticks(range(len(_t))); ax.set_yticklabels(_t.method.str.replace("graph: ", ""), fontsize=5); ax.set_xlabel("Δ macro-F1 vs same classifier on raw")
ax.set_title("Gene-graph operators", loc="left", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig_graph.png", dpi=150); graph_T

## 5. Gene pairs and gene order: is there signal beyond single genes?
(1) Amplitude-invariant gene-order representations: rank-normal scores, top-N signed genes, a signed-rank similarity (ZhangScore-like) and a
CMap-style Kolmogorov-Smirnov connectivity. (2) Top-scoring pairs chosen inside each training part, against random pairs and against the same
number of single genes chosen by the analogous criterion. (3) Pairs defined by prior knowledge (STRING ≥ 0.9, same pathway) as differences,
against random pairs. (4) All pairwise interactions implicitly: linear vs degree-2 polynomial vs RBF kernel.

In [ ]:
from scipy.stats import norm
from sklearn.svm import SVC
pairs_H = A[:, :, HI].mean(2)                                                  # top-2-dose profile, compounds x genes
_r = np.argsort(np.argsort(A, 1), 1); pairs_RN = norm.ppf((_r + 0.5) / G).astype(np.float32)    # rank-normal scores per compound and dose
evaluate_rep("pairs: rank-normal scores per profile", pairs_RN, "pairs")
for n in [50, 100, 200]:
    th = -np.sort(-np.abs(A), 1)[:, n - 1:n]; evaluate_rep(f"pairs: top-{n} signed genes per profile, rest zero", np.where(np.abs(A) >= th, A, 0).astype(np.float32), "pairs", clfs=("centroid", "LR"))

def pairs_signed_rank(X): return np.sign(X) * np.argsort(np.argsort(np.abs(X), 1), 1)
def pairs_zhang(tr, pr, n=100):
    """signed-rank similarity of the query's n most extreme genes to each class centroid (scaled to [-1, 1])"""
    C = np.stack([pairs_H[tr][yi_all[tr] == c].mean(0) for c in range(K)]); q, rc = pairs_signed_rank(pairs_H[pr]), pairs_signed_rank(C)
    top = np.argsort(-np.abs(q), 1)[:, :n]; qs = np.take_along_axis(q, top, 1)
    sc = np.stack([(qs[i] * rc[:, top[i]]).sum(1) for i in range(len(pr))])      # pr x classes
    return softmax(CEN_TEMP * sc / (np.arange(G - n, G) ** 2).sum())
def pairs_ks(tr, pr, n=50):
    """CMap 2006 style: KS enrichment of the query's n up and n down genes in each class centroid's ranking; score = ES_up - ES_down"""
    C = np.stack([pairs_H[tr][yi_all[tr] == c].mean(0) for c in range(K)]); pos = np.argsort(np.argsort(-C, 1), 1) + 1     # 1-based rank of each gene per class
    def es(V):
        V = np.sort(V, -1); t = V.shape[-1]; j = np.arange(1, t + 1); a = (j / t - V / G).max(-1); b = (V / G - (j - 1) / t).max(-1); return np.where(a > b, a, -b)
    o = np.argsort(-pairs_H[pr], 1); up, dn = o[:, :n], o[:, -n:]
    sc = np.stack([es(pos[:, u]) - es(pos[:, d]) for u, d in zip(up, dn)]); return softmax(CEN_TEMP * sc)
evaluate("pairs: signed-rank similarity, top-100 genes (ZhangScore-like) | centroid", pairs_zhang, "pairs", "raw | centroid")
evaluate("pairs: KS connectivity, top-50 up / down genes (CMap 2006 style) | centroid", pairs_ks, "pairs", "raw | centroid")

**Top-scoring pairs (k-TSP)**: inside each training part, for every class the k gene pairs (among the 150 most variable genes, a label-free choice)
with the largest |P(z_i > z_j | class) − P(z_i > z_j | rest)|. Features: the pair indicators, classified by LR, or a k-TSP vote. Controls: the same
number of random pairs (3 draws), and the same number of single genes chosen per class by the analogous mean-difference criterion. All of these
use the top-2-dose profile, so their reference is LR on that profile (section `pairs_ctrl`).

In [ ]:
pairs_V = np.argsort(-pairs_H.var(0))[:150]; pairs_I, pairs_J = np.triu_indices(150, 1); pairs_I, pairs_J = pairs_V[pairs_I], pairs_V[pairs_J]
def pairs_lr(F, tr, pr): return clf_lr(F[:, :, None], tr, pr)
evaluate("pairs ref: top-2-dose profile | LR", lambda tr, pr: pairs_lr(pairs_H, tr, pr), "pairs_ctrl")
def pairs_tsp(tr, pr, k=10, vote=False):
    B = (pairs_H[:, pairs_I] > pairs_H[:, pairs_J]).astype(np.float32); y = yi_all[tr]; chosen, sgn = [], []
    for c in range(K):
        d = B[tr][y == c].mean(0) - B[tr][y != c].mean(0); t = np.argsort(-np.abs(d))[:k]; chosen.append(t); sgn.append(np.sign(d[t]))
    if vote: return softmax(5 * np.stack([((B[pr][:, t] == (s > 0)) * 1.0).mean(1) for t, s in zip(chosen, sgn)], 1))
    return pairs_lr(B[:, np.unique(np.concatenate(chosen))], tr, pr)
def pairs_single(tr, pr, k=10):
    y = yi_all[tr]; Z = (pairs_H - pairs_H[tr].mean(0)) / (pairs_H[tr].std(0) + 1e-6)
    g = np.unique(np.concatenate([np.argsort(-np.abs(Z[tr][y == c].mean(0) - Z[tr][y != c].mean(0)))[:k] for c in range(K)])); return pairs_lr(pairs_H[:, g], tr, pr)
for k in [5, 20]:
    evaluate(f"pairs: top-scoring pairs, k={k} per class | LR", lambda tr, pr, k=k: pairs_tsp(tr, pr, k), "pairs", "pairs ref: top-2-dose profile | LR")
    evaluate(f"pairs: top-scoring pairs, k={k} per class | k-TSP vote", lambda tr, pr, k=k: pairs_tsp(tr, pr, k, True), "pairs", "pairs ref: top-2-dose profile | LR")
    evaluate(f"pairs: single genes, k={k} per class (same criterion) | LR", lambda tr, pr, k=k: pairs_single(tr, pr, k), "pairs", "pairs ref: top-2-dose profile | LR")
    for j in range(3):
        rp = np.random.default_rng(SEED + 500 + 10 * k + j).choice(len(pairs_I), min(K * k, len(pairs_I)), replace=False)
        evaluate(f"pairs ctrl: {K * k} random pairs (indicators) #{j} | LR", lambda tr, pr, rp=rp: pairs_lr((pairs_H[:, pairs_I[rp]] > pairs_H[:, pairs_J[rp]]).astype(np.float32), tr, pr), "pairs_ctrl", f"pairs: top-scoring pairs, k={k} per class | LR")

**Prior-defined pairs** as differences z_i − z_j of the top-2-dose profile: STRING edges with score ≥ 0.9 and as many random same-pathway pairs,
against the same number of random gene pairs (3 draws). **Kernels** on the standardised gene × dose features (C = 1, fixed a priori).

In [ ]:
_gi = {g: i for i, g in enumerate(genes)}; _E = STRING[STRING.score >= 0.9] if STRING is not None else STRING
pairs_prior = {"STRING >= 0.9 edges": (_E.probe_a.map(_gi).values, _E.probe_b.map(_gi).values)}
_sp = np.array([(a, b) for ix in sets.values() for a in ix for b in ix if a < b]); _sp = np.unique(_sp, axis=0)
_sp = _sp[np.random.default_rng(SEED + 530).choice(len(_sp), len(_E), replace=False)]; pairs_prior["same-pathway pairs"] = (_sp[:, 0], _sp[:, 1])
for nm, (a, b) in pairs_prior.items():
    evaluate(f"pairs: {nm} ({len(a)}), differences | LR", lambda tr, pr, a=a, b=b: pairs_lr(pairs_H[:, a] - pairs_H[:, b], tr, pr), "pairs", "pairs ref: top-2-dose profile | LR")
    for j in range(3):
        r = np.random.default_rng(SEED + 540 + j); ra, rb = r.integers(G, size=len(a)), r.integers(G, size=len(a))
        evaluate(f"pairs ctrl: {len(a)} random pairs, differences #{j} ({nm}) | LR", lambda tr, pr, ra=ra, rb=rb: pairs_lr(pairs_H[:, ra] - pairs_H[:, rb], tr, pr), "pairs_ctrl", f"pairs: {nm} ({len(a)}), differences | LR")
def pairs_svc(tr, pr, kernel):
    Ftr, Fp = std_pair(A[tr].reshape(len(tr), -1), A[pr].reshape(len(pr), -1))
    m = SVC(kernel=kernel, C=1.0, degree=2, gamma="scale", coef0=1.0, class_weight="balanced", decision_function_shape="ovr").fit(Ftr, yi_all[tr])
    P = np.full((len(pr), K), -9.0); P[:, m.classes_] = m.decision_function(Fp); return softmax(3 * P)
evaluate("pairs: SVM, linear kernel | SVM", lambda tr, pr: pairs_svc(tr, pr, "linear"), "pairs", "raw | LR")
for kn in ["poly", "rbf"]: evaluate(f"pairs: SVM, {'degree-2 polynomial' if kn == 'poly' else 'RBF'} kernel (pairwise interactions) | SVM", lambda tr, pr, kn=kn: pairs_svc(tr, pr, kn), "pairs", "pairs: SVM, linear kernel | SVM")
_c = table("pairs_ctrl"); _c = _c[_c.method.str.contains("random")]
print("random-pair controls, mean seeded-CV macro-F1:", _c.groupby(_c.method.str.replace(r" #\d+", "", regex=True)).cv_mean.mean().round(3).to_dict())

In [ ]:
pairs_T = save("pairs", "pairs_results.csv")
_t = table("pairs"); _t = _t[_t.vs.notna()].sort_values("diff").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5)); ax.hlines(range(len(_t)), _t.lo, _t.hi, color="grey"); ax.scatter(_t["diff"], range(len(_t)), c=_t.verdict.map({"better": "g", "worse": "r"}).fillna("k"), s=12, zorder=3)
ax.axvline(0, color="k", lw=.6); ax.set_yticks(range(len(_t))); ax.set_yticklabels(_t.method.str.replace("pairs: ", "") + "  [vs " + _t.vs.str.replace("pairs ref: ", "") + "]", fontsize=5)
ax.set_xlabel("Δ macro-F1 vs reference"); ax.set_title("Gene pairs and gene order", loc="left", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig_pairs.png", dpi=150); pairs_T

## 6. Easy vs hard classes: re-weighting and two-stage prediction
The idea: (1) predict the easy classes plus **one merged HARD group**, then (2) sub-classify the hard group with a model trained only on hard
classes; alternatively, re-weight training so that easy samples get less focus. Literature: flat versus hierarchical classification gives mixed
results (Babbar et al. 2013); sample re-weighting fades in over-parameterised models (Byrd & Lipton 2019).

**Declared a priori, from the biology in the challenge brief and before any result below:** in KRAS-mutant A549, EGFR, RAF, JAK, PARP and PI3K
inhibitors are expected to be quiet (variant: plus AKT). A second hard set is learned inside each training part by inner CV (classes with inner
recall below 0.5), and a third route is label-free: silent compounds.

In [ ]:
hard_PRIOR = [c for c in classes if c.split()[0] in ("EGFR", "RAF", "JAK", "PARP", "PI3K")]
hard_PRIOR_AKT = hard_PRIOR + ["AKT inhibitor"]
_rows = []
for m in ["raw | centroid", "raw | LR", "raw | reverse", "raw | ensemble"]:
    pc = calls(m)[1:]
    for c in range(K):
        _rows.append(dict(model=m, moa=classes[c], recall=(pc[:, yi == c] == c).mean(), precision=(yi[np.nonzero(pc == c)[1]] == c).mean() if (pc == c).any() else np.nan,
                          called_as=classes[np.bincount(pc[:, yi == c].ravel(), minlength=K).argmax()]))
hard_D = pd.DataFrame(_rows).pivot_table(index="moa", columns="model", values=["recall", "precision"]).round(2)
hard_D[("", "n_compounds")] = np.bincount(yi, minlength=K); hard_D[("", "share_silent")] = [silent[TR][yi == c].mean().round(2) for c in range(K)]
hard_D[("", "a_priori_hard")] = [c in hard_PRIOR for c in classes]; hard_D.to_csv(OUT / "hard_per_class.csv"); hard_D.sort_values(("recall", "raw | ensemble"))

**Two-stage model.** Stage 1: a classifier on merged labels (easy classes + HARD). Stage 2: trained on hard-class compounds only. Soft combination
P(c) = P1(c) for easy c and P1(HARD) · P2(c | HARD) for hard c; hard routing sends a compound to stage 2 only if HARD is its stage-1 argmax.
**Oracle ceilings** (section `hard_oracle`, use true labels of the predicted compounds on purpose, so they bound what the design could gain):
a perfect stage 1, and a perfect stage 2.
Note on the soft combination: P1(HARD) is shared among all hard classes, so each hard class's probability is a fraction of it and the hard
classes are under-called whenever an easy class has moderate probability; hard routing does not have this problem.

In [ ]:
def hard_inner_set(tr, thr=0.5):
    """hard classes learned on tr only: inner 3-fold CV recall of the centroid below thr"""
    y = yi_all[tr]; f = np.zeros(len(tr), int); f[np.argsort(np.random.default_rng(len(tr)).random(len(tr)))] = np.arange(len(tr)) % 3; pred = np.zeros(len(tr), int)
    for o in range(3): pred[f == o] = clf_centroid(A, tr[f != o], tr[f == o]).argmax(1)
    return [classes[c] for c in np.unique(y) if np.mean(pred[y == c] == c) < thr]
def hard_two_stage(tr, pr, hard, s1="centroid", s2="centroid", route="soft", oracle=None, R2=None):
    hard = hard(tr) if callable(hard) else hard; H = np.sort([classes.index(c) for c in hard]); E_ = np.setdiff1d(np.arange(K), H)
    if len(H) < 2: return CLF[s1](A, tr, pr)
    y = yi_all[tr]; ym = np.where(np.isin(y, H), len(E_), np.searchsorted(E_, y))            # merged labels: easy classes 0..|E|-1, HARD = |E|
    P1 = CLF[s1](A, tr, pr, y=ym, k=len(E_) + 1)
    trh = tr[np.isin(y, H)]; P2 = CLF[s2](A if R2 is None else R2, trh, pr, y=np.searchsorted(H, yi_all[trh]), k=len(H))
    if oracle == "stage 1": P1 = np.eye(len(E_) + 1)[np.where(np.isin(yi_all[pr], H), len(E_), np.searchsorted(E_, np.clip(yi_all[pr], 0, None)))]
    if oracle == "stage 2": P2 = np.eye(len(H))[np.clip(np.searchsorted(H, yi_all[pr]), 0, len(H) - 1)]
    if route == "hard": P1 = np.eye(len(E_) + 1)[P1.argmax(1)]
    P = np.zeros((len(pr), K)); P[:, E_] = P1[:, :-1]; P[:, H] = P1[:, -1:] * P2; return P
for s in ["centroid", "reverse"]:
    for hn, hs in [("a-priori hard set", hard_PRIOR), ("a-priori hard set + AKT", hard_PRIOR_AKT), ("hard set learned by inner CV", hard_inner_set)]:
        evaluate(f"hard: two-stage, {hn}, soft | {s}", lambda tr, pr, hs=hs, s=s: hard_two_stage(tr, pr, hs, s, s), "hard", f"raw | {s}")
    evaluate(f"hard: two-stage, a-priori hard set, hard routing | {s}", lambda tr, pr, s=s: hard_two_stage(tr, pr, hard_PRIOR, s, s, "hard"), "hard", f"raw | {s}")
    for o in ["stage 1", "stage 2"]: evaluate(f"hard oracle: perfect {o}, a-priori hard set | {s}", lambda tr, pr, o=o, s=s: hard_two_stage(tr, pr, hard_PRIOR, s, s, oracle=o), "hard_oracle", f"raw | {s}")
hard_PW = np.unique(np.concatenate([sets[p] for c in hard_PRIOR for p in PRIORS[c]]))                 # genes of the hard classes' expected pathways
evaluate("hard: two-stage, stage 2 on all doses of the hard classes' pathway genes | reverse", lambda tr, pr: hard_two_stage(tr, pr, hard_PRIOR, "reverse", "reverse", R2=A[:, hard_PW]), "hard", "raw | reverse")
evaluate("hard: two-stage, stage 2 = LR on all doses | centroid + LR", lambda tr, pr: hard_two_stage(tr, pr, hard_PRIOR, "centroid", "LR"), "hard", "raw | centroid")

**Biological hierarchy** (declared a priori): MAPK axis (EGFR, RAF, MEK), PI3K axis (PI3K, AKT, mTOR), mitosis (PLK, tubulin, CDK),
DNA damage (topoisomerase, PARP), proteostasis (HSP, proteasome), HDAC, JAK. Group first, then the class within the group.

In [ ]:
hard_GROUPS = {"MAPK": "EGFR RAF MEK", "PI3K axis": "PI3K AKT mTOR", "mitosis": "PLK tubulin CDK", "DNA damage": "topoisomerase PARP", "proteostasis": "HSP proteasome", "HDAC": "HDAC", "JAK": "JAK"}
hard_gid = np.zeros(K, int)
for j, v in enumerate(hard_GROUPS.values()):
    for k in v.split(): hard_gid[next(i for i, c in enumerate(classes) if c.startswith(k + " "))] = j
def hard_hier(tr, pr, s="centroid"):
    y = yi_all[tr]; P1 = CLF[s](A, tr, pr, y=hard_gid[y], k=len(hard_GROUPS)); P = np.zeros((len(pr), K))
    for g in range(len(hard_GROUPS)):
        cs = np.flatnonzero(hard_gid == g)
        if len(cs) == 1: P[:, cs[0]] = P1[:, g]; continue
        trg = tr[np.isin(y, cs)]; P[:, cs] = P1[:, [g]] * CLF[s](A, trg, pr, y=np.searchsorted(cs, yi_all[trg]), k=len(cs))
    return P
for s in ["centroid", "reverse"]: evaluate(f"hard: biological hierarchy (group, then class) | {s}", lambda tr, pr, s=s: hard_hier(tr, pr, s), "hard", f"raw | {s}")

**Re-weighting** (LR only: the centroid and the reverse expert have no loss to re-weight). Focal-style weights w = (1 − p_true)^γ with p_true from
inner-CV out-of-fold predictions on the training part; class-level up-weighting (×3) of the a-priori hard classes; two rounds of boosting-style
re-weighting. **Prior shift by activity**: P(c | x, silent) ∝ P(c | x) · P(c | silent) / P(c), with class shares among silent and active
compounds estimated from the training part. **Specialists**: a one-vs-rest LR on a hard class's expected-pathway genes (all doses), blended
with weight 0.5 into the general model's score for that class; control: specialists on random genes of the same size (3 draws).

In [ ]:
def hard_ptrue(tr, w=None):
    y = yi_all[tr]; f = np.zeros(len(tr), int); f[np.argsort(np.random.default_rng(len(tr) + 1).random(len(tr)))] = np.arange(len(tr)) % 3; P = np.zeros((len(tr), K))
    for o in range(3): P[f == o] = clf_lr(A, tr[f != o], tr[f == o], sample_weight=None if w is None else w[f != o])
    return P[np.arange(len(tr)), y]
def hard_focal(tr, pr, gamma): w = (1 - hard_ptrue(tr)) ** gamma + 0.05; return clf_lr(A, tr, pr, sample_weight=w / w.mean())
def hard_boost(tr, pr, rounds=2):
    w = np.ones(len(tr))
    for _ in range(rounds): w = w * np.exp(1 - hard_ptrue(tr, w)); w /= w.mean()
    return clf_lr(A, tr, pr, sample_weight=w)
for g in [1, 2]: evaluate(f"hard: focal-style sample weights, gamma={g} | LR", lambda tr, pr, g=g: hard_focal(tr, pr, g), "hard", "raw | LR")
evaluate("hard: hard classes up-weighted x3 | LR", lambda tr, pr: clf_lr(A, tr, pr, sample_weight=np.where(np.isin(np.array(classes)[yi_all[tr]], hard_PRIOR), 3.0, 1.0)), "hard", "raw | LR")
evaluate("hard: boosting-style re-weighting, 2 rounds | LR", hard_boost, "hard", "raw | LR")
def hard_shift(tr, pr, base):
    P = base(tr, pr); y = yi_all[tr]; s = silent[tr]; prior = np.bincount(y, minlength=K) + 1.0
    ps, pa = (np.bincount(y[s], minlength=K) + 1.0) / (s.sum() + K), (np.bincount(y[~s], minlength=K) + 1.0) / ((~s).sum() + K); prior /= prior.sum()
    P = P * np.where(silent[pr][:, None], ps, pa) / prior; return P / P.sum(1, keepdims=True)
for s in ["centroid", "reverse"]: evaluate(f"hard: prior shift by activity (silent / active) | {s}", lambda tr, pr, s=s: hard_shift(tr, pr, lambda a, b: CLF[s](A, a, b)), "hard", f"raw | {s}")
def hard_specialist(tr, pr, genesets, base="centroid", w=0.5):
    P = CLF[base](A, tr, pr); y = yi_all[tr]
    for c, ix in genesets.items():
        ci = classes.index(c); Ftr, Fp = std_pair(A[tr][:, ix].reshape(len(tr), -1), A[pr][:, ix].reshape(len(pr), -1))
        P[:, ci] = (1 - w) * P[:, ci] + w * LogisticRegression(C=0.05, max_iter=2000, class_weight="balanced").fit(Ftr, y == ci).predict_proba(Fp)[:, 1]
    return P / P.sum(1, keepdims=True)
hard_spec = {c: np.unique(np.concatenate([sets[p] for p in PRIORS[c]])) for c in hard_PRIOR}
evaluate("hard: expected-pathway specialists for hard classes, blend 0.5 | centroid", lambda tr, pr: hard_specialist(tr, pr, hard_spec), "hard", "raw | centroid")
for j in range(3):
    rs = {c: np.random.default_rng(SEED + 600 + 10 * j + i).choice(G, len(ix), replace=False) for i, (c, ix) in enumerate(hard_spec.items())}
    evaluate(f"hard ctrl: random-gene specialists, blend 0.5 #{j} | centroid", lambda tr, pr, rs=rs: hard_specialist(tr, pr, rs), "hard_ctrl", "hard: expected-pathway specialists for hard classes, blend 0.5 | centroid")
_c = table("hard_ctrl"); print("random-gene specialists (3 draws): mean seeded-CV macro-F1", _c.cv_mean.mean().round(3), "| expected-pathway specialists:", table(names=["hard: expected-pathway specialists for hard classes, blend 0.5 | centroid"]).cv_mean[0])
_o = table("hard_oracle"); print(_o[["method", "cv_mean", "diff"]].to_string(index=False))

**Accepting the biology** (descriptive, `hard_merged_classes.csv`): macro-F1 of the baselines after merging classes the pathway logic says may be
inseparable, scored on merged labels. Not comparable with the 15-class numbers: fewer classes is an easier problem.

In [ ]:
hard_merges = {"none (15 classes)": [], "RAF + MEK": [["RAF", "MEK"]], "PI3K + AKT + mTOR": [["PI3K", "AKT", "mTOR"]], "whole MAPK axis (EGFR + RAF + MEK)": [["EGFR", "RAF", "MEK"]],
               "MAPK axis and PI3K axis": [["EGFR", "RAF", "MEK"], ["PI3K", "AKT", "mTOR"]]}
_rows = []
for nm, groups in hard_merges.items():
    mp = np.arange(K)
    for grp in groups: ids = [next(i for i, c in enumerate(classes) if c.startswith(k + " ")) for k in grp]; mp[ids] = ids[0]
    yt = mp[yi]; u = np.unique(mp)
    for m in ["raw | centroid", "raw | ensemble"]:
        f = [np.mean([2 * ((p == c) & (yt == c)).sum() / max((p == c).sum() + (yt == c).sum(), 1) for c in u]) for p in mp[calls(m)[1:]]]
        _rows.append(dict(merge=nm, n_classes=len(u), model=m, macro_f1=np.mean(f), sd=np.std(f)))
hard_M = pd.DataFrame(_rows).round(3); hard_M.to_csv(OUT / "hard_merged_classes.csv", index=False); print(hard_M.pivot(index="merge", columns="model", values="macro_f1").to_string())

In [ ]:
hard_T = save("hard", "hard_results.csv")
_r = hard_D[("recall", "raw | ensemble")].sort_values(); fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(range(K), _r.values, color=["tab:red" if c in hard_PRIOR else "tab:blue" for c in _r.index]); ax.set_yticks(range(K)); ax.set_yticklabels(_r.index, fontsize=7)
ax.set_xlabel("recall (seeded CV, mean of 3 base models)"); ax.set_title("Per-class recall; red = declared hard a priori (quiet in KRAS-mutant A549)", loc="left", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "fig_hard.png", dpi=150); hard_T

## 7. Unlabelled compounds: do the 1,286 extra compounds help?
Four uses, all label-free with respect to the unlabelled compounds: (1) geometry (global centring, PCA fitted on all compounds); (2) a
**dose-consistency metric**, a linear stand-in for contrastive learning: the top doses of each active unlabelled compound are treated as
replicates, so within-compound variation is noise and between-compound variation is signal; (3) **label spreading** over a compound kNN graph in
which unlabelled compounds act as bridges (long-range relations between compounds); (4) self-training and reference-percentile (tau-like) scores.
Controls: compound identities shuffled among the unlabelled signatures, the graph without unlabelled nodes or with inactive ones, random labels.

In [ ]:
unl_act = UNL[~silent[UNL]]; unl_sil = UNL[silent[UNL]]; unl_H = A[:, :, HI].mean(2)
print(f"unlabelled compounds: {len(UNL)}, active (not silent): {len(unl_act)}")
evaluate_rep("unl: global centring over all compounds", A - A.mean(0, keepdims=True), "unl")
def unl_pca(fit_rows, k):
    X = unl_H - unl_H[fit_rows].mean(0); V = np.linalg.svd(X[fit_rows], full_matrices=False)[2][:k]; return (X @ V.T)[:, :, None].astype(np.float32)
for k in [10, 20, 50, 100]: evaluate_rep(f"unl: PCA on all compounds, {k} components", unl_pca(np.arange(N), k), "unl", clfs=("centroid", "LR"))
evaluate_rep("unl ctrl: PCA on train + test compounds only, 50 components", unl_pca(np.arange(n_lab), 50), "unl_ctrl", clfs=("centroid", "LR"), ref_rep="unl: PCA on all compounds, 50 components")

**Dose-consistency metric** from the active unlabelled compounds (doses 1.11, 3.33 and 10 µM as replicates): (a) per-gene weights √(between / within
variance), then the usual classifiers; (b) whitening by the shrunk within-compound covariance in a 100-component PCA space; (c) discriminant
directions with compounds as classes (shrinkage LDA, 50 components). Control: the same with compound identities shuffled among the signatures.

In [ ]:
from sklearn.covariance import LedoitWolf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
def unl_metric(seed=None):
    X = A[unl_act][:, :, 3:].transpose(0, 2, 1).reshape(-1, G); cid = np.repeat(np.arange(len(unl_act)), 3)
    if seed is not None: cid = np.random.default_rng(seed).permutation(cid)
    M = pd.DataFrame(X).groupby(cid).transform("mean").values; within = ((X - M) ** 2).mean(0); between = pd.DataFrame(X).groupby(cid).mean().var(0).values
    w = np.sqrt(between / (within + 1e-6)); w /= np.median(w)
    mu = X.mean(0); V = np.linalg.svd(X - mu, full_matrices=False)[2][:100]; Z = (X - mu) @ V.T; Zw = Z - pd.DataFrame(Z).groupby(cid).transform("mean").values
    c_ = LedoitWolf().fit(Zw).covariance_; e, Q = np.linalg.eigh(c_); Wh = Q @ np.diag(e ** -0.5) @ Q.T
    lda = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto", n_components=50).fit(Z, cid)
    Hp = (unl_H - mu) @ V.T
    return {"(a) per-gene weights": (A * w[None, :, None]).astype(np.float32), "(b) within-compound whitening": (Hp @ Wh)[:, :, None].astype(np.float32),
            "(c) compound-discriminant directions": lda.transform(Hp)[:, :, None].astype(np.float32)}
unl_M = unl_metric()
for nm, R in unl_M.items(): evaluate_rep(f"unl: dose-consistency metric {nm}", R, "unl", clfs=("centroid", "LR", "reverse") if "(a)" in nm else ("centroid", "LR"))
for j in range(2):
    for nm, R in unl_metric(SEED + 700 + j).items():
        evaluate_rep(f"unl ctrl: compound identities shuffled #{j}, {nm}", R, "unl_ctrl", clfs=("centroid", "LR"), ref_rep=f"unl: dose-consistency metric {nm}")
_c = table("unl_ctrl"); _c = _c[_c.method.str.contains("shuffled")]
print("metric learned with shuffled compound identities (2 draws), mean seeded-CV macro-F1:", _c.groupby(_c.method.str.replace(r" #\d+", "", regex=True).str.replace("unl ctrl: compound identities shuffled, ", "")).cv_mean.mean().round(3).to_dict())

**Label spreading** (closed form F = (I − αS)⁻¹Y, Zhou et al. 2004) on a symmetric kNN graph (k = 10, Pearson r of the top-2-dose profiles)
over the training compounds, the compounds to predict and the active unlabelled compounds; seeded only with the training labels.

In [ ]:
def unl_spread(tr, pr, extra, alpha=0.9, k=10):
    nodes = np.concatenate([tr, pr, extra]); C = corr_rows(unl_H[nodes], unl_H[nodes]); np.fill_diagonal(C, -9)
    W = np.zeros_like(C); nn = np.argsort(-C, 1)[:, :k]; W[np.repeat(np.arange(len(nodes)), k), nn.ravel()] = np.clip(C[np.repeat(np.arange(len(nodes)), k), nn.ravel()], 0, None)
    W = np.maximum(W, W.T); d = W.sum(1) ** -0.5; d[~np.isfinite(d)] = 0; Sn = d[:, None] * W * d[None]
    Y = np.zeros((len(nodes), K)); Y[np.arange(len(tr)), yi_all[tr]] = 1; Y[:len(tr)] /= Y[:len(tr)].sum(0, keepdims=True) + 1e-9      # class-balanced seeds
    F = np.linalg.solve(np.eye(len(nodes)) - alpha * Sn, Y)[len(tr):len(tr) + len(pr)] + 1e-9; return F / F.sum(1, keepdims=True)
for a in [0.5, 0.9]: evaluate(f"unl: label spreading over train + test + active unlabelled, alpha={a} | spreading", lambda tr, pr, a=a: unl_spread(tr, pr, unl_act, a), "unl", "raw | centroid")
evaluate("unl ctrl: label spreading without unlabelled nodes, alpha=0.9 | spreading", lambda tr, pr: unl_spread(tr, pr, np.array([], int)), "unl_ctrl", "unl: label spreading over train + test + active unlabelled, alpha=0.9 | spreading")
_ri = np.random.default_rng(SEED + 710).choice(unl_sil, min(len(unl_act), len(unl_sil)), replace=False)
evaluate(f"unl ctrl: label spreading with {len(_ri)} inactive unlabelled nodes instead, alpha=0.9 | spreading", lambda tr, pr: unl_spread(tr, pr, _ri), "unl_ctrl", "unl: label spreading over train + test + active unlabelled, alpha=0.9 | spreading")

**Self-training**: an active unlabelled compound joins a class if its correlation with that class's centroid exceeds 0.5 and beats the second
class by 0.1 (thresholds fixed a priori); centroids are then recomputed. Control: the same number of active unlabelled compounds per class, drawn
at random. **Tau-like scores**: each compound-to-centroid correlation expressed as a percentile among the unlabelled compounds' correlations with
that centroid.

In [ ]:
def unl_selftrain(tr, pr, random_seed=None, thr=0.5, margin=0.1, report=False):
    y = yi_all[tr]; C = np.stack([unl_H[tr][y == c].mean(0) for c in range(K)]); r = corr_rows(unl_H[unl_act], C); s = np.sort(r, 1)
    ok = (s[:, -1] > thr) & (s[:, -1] - s[:, -2] > margin); add, lab = unl_act[ok], r[ok].argmax(1)
    if random_seed is not None: add = np.random.default_rng(random_seed).choice(unl_act, len(add), replace=False)
    if report: return pd.Series(np.bincount(lab, minlength=K), index=classes)
    C2 = np.stack([np.vstack([unl_H[tr][y == c], unl_H[add][lab == c]]).mean(0) for c in range(K)]); return softmax(CEN_TEMP * corr_rows(unl_H[pr], C2))
evaluate("unl: self-training, centroids enlarged with confident unlabelled compounds | centroid", unl_selftrain, "unl", "raw | centroid")
for j in range(3): evaluate(f"unl ctrl: self-training with random unlabelled compounds #{j} | centroid", lambda tr, pr, j=j: unl_selftrain(tr, pr, SEED + 720 + j), "unl_ctrl", "unl: self-training, centroids enlarged with confident unlabelled compounds | centroid")
print("unlabelled compounds added per class when fitted on all training compounds:", unl_selftrain(TR, TE, report=True).to_dict())
def unl_tau(tr, pr):
    y = yi_all[tr]; C = np.stack([unl_H[tr][y == c].mean(0) for c in range(K)]); ref = np.sort(corr_rows(unl_H[UNL], C), 0); r = corr_rows(unl_H[pr], C)
    pct = np.stack([np.searchsorted(ref[:, c], r[:, c]) / len(UNL) for c in range(K)], 1); return softmax(CEN_TEMP * pct)
evaluate("unl: tau-like percentile scores against the unlabelled compounds | centroid", unl_tau, "unl", "raw | centroid")

In [ ]:
unl_T = save("unl", "unl_results.csv")
_t = table("unl"); _t = _t[_t.vs.notna()].sort_values("diff").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5.5)); ax.hlines(range(len(_t)), _t.lo, _t.hi, color="grey"); ax.scatter(_t["diff"], range(len(_t)), c=_t.verdict.map({"better": "g", "worse": "r"}).fillna("k"), s=12, zorder=3)
ax.axvline(0, color="k", lw=.6); ax.set_yticks(range(len(_t))); ax.set_yticklabels(_t.method.str.replace("unl: ", ""), fontsize=5); ax.set_xlabel("Δ macro-F1 vs same classifier on raw")
ax.set_title("Using the unlabelled compounds", loc="left", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig_unl.png", dpi=150); unl_T

## 8. Other classifiers, preprocessing, ensembling, and honest model selection
Few-shot classifiers (nearest shrunken centroids, shrinkage LDA, kNN, nearest compound), standard models (linear SVM, ridge, naive Bayes, tree
ensembles), Kaggle-MoA preprocessing, centroid variants and ways to combine models. Hyperparameters are fixed a priori, or chosen inside each
training part by inner CV. Unless stated, models use the top-2-dose profile, so their reference is LR on that profile (section `zoo_ctrl`).
Finally, a nested "pick the winner" shows how much choosing the best row on these 87 compounds is worth, and a check of whether confidence is informative.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.preprocessing import QuantileTransformer
from scipy.stats import rankdata
zoo_H = A[:, :, HI].mean(2)
evaluate("zoo ref: top-2-dose profile | LR", lambda tr, pr: clf_lr(zoo_H[:, :, None], tr, pr), "zoo_ctrl")
def zoo_full(P, classes_, n):
    out = np.zeros((n, K)); out[:, classes_] = P; return out
def zoo_nsc(tr, pr, delta=0.0, X=zoo_H):
    """nearest shrunken centroids (Tibshirani et al. 2002); delta = 0 is diagonal LDA"""
    y = yi_all[tr]; Xt = X[tr]; n = len(tr); mu = Xt.mean(0); cs = np.unique(y); C = np.stack([Xt[y == c].mean(0) for c in cs])
    s = np.sqrt(sum(((Xt[y == c] - C[i]) ** 2).sum(0) for i, c in enumerate(cs)) / (n - len(cs))); s0 = np.median(s)
    m = np.sqrt(1 / np.array([(y == c).sum() for c in cs]) - 1 / n)[:, None]; d = (C - mu) / (m * (s + s0)); d = np.sign(d) * np.clip(np.abs(d) - delta, 0, None)
    Cs = mu + m * (s + s0) * d; D = (((X[pr][:, None] - Cs[None]) / (s + s0)) ** 2).sum(2); return zoo_full(softmax(-0.5 * D / np.sqrt(G)), cs, len(pr))
def zoo_inner(tr, fn, cands, k=3):
    """choose among candidate settings by inner k-fold CV on tr only (balanced accuracy)"""
    y = yi_all[tr]; f = np.zeros(len(tr), int); f[np.argsort(np.random.default_rng(len(tr) + 7).random(len(tr)))] = np.arange(len(tr)) % k; sc = []
    for c in cands:
        p = np.zeros(len(tr), int)
        for o in range(k): p[f == o] = fn(tr[f != o], tr[f == o], c).argmax(1)
        sc.append(np.mean([np.mean(p[y == u] == u) for u in np.unique(y)]))
    return cands[int(np.argmax(sc))]
for d in [0, 1, 2, 4]: evaluate(f"zoo: nearest shrunken centroids, delta={d}{' (diagonal LDA)' if d == 0 else ''} | NSC", lambda tr, pr, d=d: zoo_nsc(tr, pr, d), "zoo", "raw | centroid")
evaluate("zoo: nearest shrunken centroids, delta by inner CV | NSC", lambda tr, pr: zoo_nsc(tr, pr, zoo_inner(tr, zoo_nsc, [0, 0.5, 1, 2, 4])), "zoo", "raw | centroid")
def zoo_sk(model, tr, pr, X=zoo_H, std=True):
    Ftr, Fp = std_pair(X[tr], X[pr]) if std else (X[tr], X[pr]); m = model.fit(Ftr, yi_all[tr])
    if hasattr(m, "predict_proba"): return zoo_full(m.predict_proba(Fp), m.classes_, len(pr))
    return zoo_full(softmax(3 * m.decision_function(Fp)), m.classes_, len(pr))
zoo_models = {"LDA, Ledoit-Wolf shrinkage": lambda: LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), "linear SVM (C=0.01)": lambda: LinearSVC(C=0.01, class_weight="balanced", max_iter=5000),
              "ridge classifier (alpha=100)": lambda: RidgeClassifier(alpha=100.0, class_weight="balanced"), "Gaussian naive Bayes": lambda: GaussianNB()}
for nm, mk in zoo_models.items(): evaluate(f"zoo: {nm} | {nm.split(',')[0].split(' (')[0]}", lambda tr, pr, mk=mk: zoo_sk(mk(), tr, pr), "zoo", "zoo ref: top-2-dose profile | LR")
for s in range(3): evaluate(f"zoo: extra trees (500 trees), seed {s} | trees", lambda tr, pr, s=s: zoo_sk(ExtraTreesClassifier(500, class_weight="balanced", random_state=s, n_jobs=1), tr, pr, std=False), "zoo", "zoo ref: top-2-dose profile | LR")
evaluate("zoo: random forest (500 trees), seed 0 | trees", lambda tr, pr: zoo_sk(RandomForestClassifier(500, class_weight="balanced", random_state=0, n_jobs=1), tr, pr, std=False), "zoo", "zoo ref: top-2-dose profile | LR")
def zoo_knn(tr, pr, k=3, nearest=False):
    C = corr_rows(zoo_H[pr], zoo_H[tr]); y = yi_all[tr]
    if nearest: return softmax(CEN_TEMP * np.stack([C[:, y == c].max(1) for c in range(K)], 1))
    nn = np.argsort(-C, 1)[:, :k]; P = np.zeros((len(pr), K)) + 1e-3
    for i in range(len(pr)): np.add.at(P[i], y[nn[i]], np.clip(C[i, nn[i]], 0, None) + 1e-3)
    return P / P.sum(1, keepdims=True)
for k in [1, 3, 5]: evaluate(f"zoo: kNN, correlation, k={k} | kNN", lambda tr, pr, k=k: zoo_knn(tr, pr, k), "zoo", "raw | centroid")
evaluate("zoo: nearest training compound of each class (max r) | kNN", lambda tr, pr: zoo_knn(tr, pr, nearest=True), "zoo", "raw | centroid")

**Preprocessing and centroid variants.** Quantile-normal transform per gene × dose feature, fitted on all compounds (label-free), and PCA
components of all compounds appended (both from the Kaggle MoA solutions). Centroids with the median instead of the mean, and Spearman or cosine
instead of Pearson similarity.

In [ ]:
_Q = QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED).fit_transform(A.reshape(N, -1)).reshape(A.shape).astype(np.float32)
evaluate_rep("zoo: quantile-normal transform (fitted on all compounds)", _Q, "zoo")
_V = np.linalg.svd(zoo_H - zoo_H.mean(0), full_matrices=False)[2][:50]; zoo_PC = ((zoo_H - zoo_H.mean(0)) @ _V.T).astype(np.float32)
evaluate("zoo: raw + 50 PCA components of all compounds appended | LR", lambda tr, pr: clf_lr(np.concatenate([A.reshape(N, -1), np.repeat(zoo_PC, 6, 1)], 1)[:, :, None], tr, pr), "zoo", "raw | LR")
def zoo_cen(tr, pr, stat="mean", sim="pearson"):
    y = yi_all[tr]; C = np.stack([(np.median if stat == "median" else np.mean)(zoo_H[tr][y == c], 0) for c in range(K)]); Q = zoo_H[pr]
    if sim == "spearman": Q, C = rankdata(Q, axis=1), rankdata(C, axis=1)
    if sim == "cosine": return softmax(CEN_TEMP * (Q / np.linalg.norm(Q, axis=1, keepdims=True)) @ (C / np.linalg.norm(C, axis=1, keepdims=True)).T)
    return softmax(CEN_TEMP * corr_rows(Q, C))
for st, sm in [("median", "pearson"), ("mean", "spearman"), ("mean", "cosine")]: evaluate(f"zoo: centroid, {st}, {sm} | centroid", lambda tr, pr, st=st, sm=sm: zoo_cen(tr, pr, st, sm), "zoo", "raw | centroid")

**Ensembling** the three base models (centroid, LR, reverse): mean of probabilities (the baseline ensemble), geometric mean, rank average, majority
vote; and the mean over a wider zoo (base three + shrunken centroids δ = 2 + shrinkage LDA + kNN k = 3). **Honest selection**: inside each
training part, an inner 3-fold CV picks one of seven methods, which is then applied to the held-out fold. Compare it with the post-hoc best row.

In [ ]:
_B = [PRED[f"raw | {c}"] for c in ["centroid", "LR", "reverse"]]
evaluate("zoo: base three, geometric mean | ensemble", None, "zoo", "raw | ensemble", P=softmax(np.mean([np.log(np.clip(p, 1e-6, 1)) for p in _B], 0).reshape(-1, K)).reshape(_B[0].shape))
evaluate("zoo: base three, rank average | ensemble", None, "zoo", "raw | ensemble", P=np.mean([np.argsort(np.argsort(p, -1), -1) for p in _B], 0) / K)
_v = np.mean([np.eye(K)[p.argmax(-1)] for p in _B], 0) + 1e-3 * np.mean(_B, 0); evaluate("zoo: base three, majority vote | ensemble", None, "zoo", "raw | ensemble", P=_v)
ensemble("zoo: wider zoo (base three + NSC + LDA + kNN), mean | ensemble", ["raw | centroid", "raw | LR", "raw | reverse", "zoo: nearest shrunken centroids, delta=2 | NSC",
         "zoo: LDA, Ledoit-Wolf shrinkage | LDA", "zoo: kNN, correlation, k=3 | kNN"], "zoo", "raw | ensemble")
zoo_SHORT = {"centroid": lambda tr, pr: clf_centroid(A, tr, pr), "LR": lambda tr, pr: clf_lr(A, tr, pr), "reverse": lambda tr, pr: clf_reverse(A, tr, pr),
             "ensemble": lambda tr, pr: np.mean([clf_centroid(A, tr, pr), clf_lr(A, tr, pr), clf_reverse(A, tr, pr)], 0),
             "NSC delta=2": lambda tr, pr: zoo_nsc(tr, pr, 2), "LDA": lambda tr, pr: zoo_sk(LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), tr, pr), "kNN k=3": lambda tr, pr: zoo_knn(tr, pr, 3)}
zoo_picked = []
def zoo_nested(tr, pr):
    best = zoo_inner(tr, lambda a, b, nm: zoo_SHORT[nm](a, b), list(zoo_SHORT)); zoo_picked.append(best); return zoo_SHORT[best](tr, pr)
evaluate("zoo: nested CV picks one of 7 methods per training part | nested", zoo_nested, "zoo", "raw | centroid")
print("methods picked by the nested selection:", pd.Series(zoo_picked).value_counts().to_dict())

**Is confidence informative?** (descriptive, `zoo_confidence.csv`) For the base-three ensemble over the seeded splits: AUC of "call is correct"
from the top probability and from the margin to the second class, accuracy by confidence tercile, and for active vs silent compounds.

In [ ]:
from sklearn.metrics import roc_auc_score
_P = PRED["raw | ensemble"][1:]; _ok = (_P.argmax(-1) == yi[None]); _top = np.sort(_P, -1); _conf, _marg = _top[..., -1], _top[..., -1] - _top[..., -2]
_t = np.quantile(_conf, [1 / 3, 2 / 3]); _ter = np.digitize(_conf, _t)
zoo_CONF = pd.DataFrame([dict(measure="AUC of correctness from top probability", value=roc_auc_score(_ok.ravel(), _conf.ravel())),
                         dict(measure="AUC of correctness from margin", value=roc_auc_score(_ok.ravel(), _marg.ravel()))] +
                        [dict(measure=f"accuracy, confidence tercile {i + 1} (low to high)", value=_ok[_ter == i].mean()) for i in range(3)] +
                        [dict(measure="accuracy, active compounds", value=_ok[:, ~silent[TR]].mean()), dict(measure="accuracy, silent compounds", value=_ok[:, silent[TR]].mean()),
                         dict(measure="AUC of correctness from top probability, active compounds only", value=roc_auc_score(_ok[:, ~silent[TR]].ravel(), _conf[:, ~silent[TR]].ravel()))]).round(3)
zoo_CONF.to_csv(OUT / "zoo_confidence.csv", index=False); zoo_CONF

In [ ]:
zoo_T = save("zoo", "zoo_results.csv")
_a = table("zoo"); _best = _a.loc[_a.cv_mean.idxmax()]; _n = table(names=["zoo: nested CV picks one of 7 methods per training part | nested"]).cv_mean[0]
print(f"post-hoc best row: {_best.method} ({_best.cv_mean:.3f}); nested selection: {_n:.3f}; centroid: {table(names=['raw | centroid']).cv_mean[0]:.3f}")
_t = _a.sort_values("cv_mean").reset_index(drop=True); fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(_t.cv_mean, range(len(_t)), xerr=_t.cv_sd, fmt="o", ms=3, color="k"); ax.axvline(table(names=["raw | centroid"]).cv_mean[0], ls="--", color="grey", lw=.8)
ax.set_yticks(range(len(_t))); ax.set_yticklabels(_t.method.str.replace("zoo: ", ""), fontsize=5); ax.set_xlabel("seeded-CV macro-F1 (± SD over seeds); dashed: centroid")
ax.set_title("Classifier zoo", loc="left", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig_zoo.png", dpi=150); zoo_T

## Summary: what worked and what did not
Every method of every section in one table (`summary_all_methods.csv`). Two readings of the same comparisons:

- `verdict`: the 95% interval of the paired difference excludes 0 (one comparison at a time);
- `familywise`: corrected for the number of comparisons made in this notebook (max-statistic over the same bootstrap resamples). With this many comparisons several single-comparison verdicts are expected by chance, so only `familywise` verdicts should be treated as findings.

Control draws (sections ending in `_ctrl`) and oracle ceilings (`_oracle`, which use true labels on purpose) are listed in their own sections above and left out here.

In [ ]:
SUMMARY = table(); SUMMARY = SUMMARY[~SUMMARY.section.str.endswith(("_ctrl", "_oracle"))].reset_index(drop=True)
_c = SUMMARY.dropna(subset=["vs"]).index
_D = np.stack([(macro_f1(calls(SUMMARY.method[i])[1:], _BW) - macro_f1(calls(SUMMARY.vs[i])[1:], _BW)).mean(1) for i in _c], 1)   # bootstrap x comparisons
_se = _D.std(0); _ok = _se > 1e-9; _z = np.where(_ok, SUMMARY.loc[_c, "diff"].values / np.where(_ok, _se, 1), 0)
FW_CRIT = np.percentile(np.abs((_D[:, _ok] - _D[:, _ok].mean(0)) / _se[_ok]).max(1), 95)                  # family-wise 5% critical value
SUMMARY.loc[_c, "z"] = _z.round(2); SUMMARY.loc[_c, "familywise"] = np.where(np.abs(_z) > FW_CRIT, np.where(_z > 0, "better", "worse"), "no evidence")
SUMMARY.to_csv(OUT / "summary_all_methods.csv", index=False)
_n = len(_c); print(f"{len(SUMMARY)} methods, {_n} paired comparisons. Single-comparison verdicts: {(SUMMARY.verdict == 'better').sum()} better, {(SUMMARY.verdict == 'worse').sum()} worse "
                    f"(about {0.025 * _n:.0f} of each expected by chance alone). Family-wise (|z| > {FW_CRIT:.2f}): {(SUMMARY.familywise == 'better').sum()} better, {(SUMMARY.familywise == 'worse').sum()} worse.")
BY = SUMMARY.dropna(subset=["vs"]).groupby("section", sort=False).apply(lambda d: pd.Series({
    "comparisons": len(d), "better": (d.verdict == "better").sum(), "worse": (d.verdict == "worse").sum(), "better_familywise": (d.familywise == "better").sum(), "worse_familywise": (d.familywise == "worse").sum(),
    "median_diff": d["diff"].median(), "largest_gain": d.loc[d["diff"].idxmax(), "method"], "its_diff": d["diff"].max(), "its_interval": f"[{d.loc[d['diff'].idxmax(), 'lo']:+.3f}, {d.loc[d['diff'].idxmax(), 'hi']:+.3f}]"}))
BY.to_csv(OUT / "summary_by_section.csv"); BY

In [ ]:
# every paired comparison, by section: difference to its like-for-like reference with the 95% interval
_secs = [s for s in SUMMARY.section.unique() if SUMMARY[SUMMARY.section == s].vs.notna().any()]
fig, axes = plt.subplots(1, len(_secs), figsize=(2.6 * len(_secs), 6), sharex=True)
for ax, s in zip(np.atleast_1d(axes), _secs):
    d = SUMMARY[(SUMMARY.section == s) & SUMMARY.vs.notna()].sort_values("diff").reset_index(drop=True)
    col = d.familywise.map({"better": "tab:green", "worse": "tab:red"}).fillna(d.verdict.map({"better": "tab:olive", "worse": "tab:orange"})).fillna("grey")
    ax.hlines(d.index, d.lo, d.hi, color=col, lw=1); ax.scatter(d["diff"], d.index, c=col, s=10, zorder=3); ax.axvline(0, color="k", lw=.6)
    ax.set_yticks([]); ax.set_title(f"{s} ({len(d)})", fontsize=9); ax.set_xlabel("Δ macro-F1 vs reference", fontsize=8)
fig.suptitle("All paired comparisons (green / red: better / worse after family-wise correction; olive / orange: single-comparison only; grey: no evidence)", x=0.01, ha="left", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "fig_summary.png", dpi=150)

In [ ]:
# leaderboard by seeded-CV macro-F1, every row also compared with the plain centroid. The top of a long list is optimistic:
# the nested "pick the winner" row of section 8 shows what selecting on these 87 compounds is really worth.
REF = "raw | centroid"
TOP = SUMMARY.sort_values("cv_mean", ascending=False).head(20)[["section", "method", "official", "cv_mean", "cv_sd", "cv_cov", "top3"]].reset_index(drop=True)
_cmp = pd.DataFrame([compare(m, REF) if m != REF else dict(diff=0, lo=0, hi=0, seeds_better="-") for m in TOP.method]).round(3)
TOP = pd.concat([TOP, _cmp.add_suffix("_vs_centroid")], axis=1); TOP.to_csv(OUT / "summary_leaderboard.csv", index=False); TOP

## Candidate model and test calls (no submission is written unless `WRITE_SUBMISSION = True`)
`CANDIDATE` is fixed here by name and is not picked from the leaderboard. `silent` = fewer genes with |z| > 2 at the two top doses than twice the noise floor: such compounds look close to untreated and their call is a guess.

In [ ]:
CANDIDATE, WRITE_SUBMISSION = ["raw | centroid", "raw | LR", "raw | reverse"], False     # members are averaged; any method name evaluated above can be used
P_te = np.mean([predict_test(INFO[m]["fn"]) for m in CANDIDATE], 0); evaluate("candidate", None, "candidate", REF, P=np.mean([PRED[m] for m in CANDIDATE], 0))
_pred = calls("candidate")[1:]; _rec = np.array([(_pred[:, yi == c] == c).mean() for c in range(K)])                 # per-class recall over the seeded splits
CALLS = pd.DataFrame({"call": np.array(classes)[P_te.argmax(1)], "call_with_coverage": np.array(classes)[cover(P_te)], "top_prob": P_te.max(1).round(3),
                      "members_agreeing": sum((predict_test(INFO[m]["fn"]).argmax(1) == P_te.argmax(1)).astype(int) for m in CANDIDATE),
                      "n_DEG_top_doses": n_deg[TE][:, HI].mean(1).round(0).astype(int), "silent": silent[TE], "cv_recall_of_called_class": _rec[P_te.argmax(1)].round(2)}, index=pd.Index(np.array(cpds)[TE], name="compound"))
CALLS.sort_values("n_DEG_top_doses").to_csv(OUT / "test_calls.csv")
_ok = (_pred == yi).mean(0); print(table(names=["candidate"]).drop(columns="section").to_string(index=False))
print(f"\nCV accuracy of the candidate: {_ok[~silent[TR]].mean():.2f} on {(~silent[TR]).sum()} active training compounds, {_ok[silent[TR]].mean():.2f} on {silent[TR].sum()} silent ones; {silent[TE].sum()} of {n_te} test compounds are silent")
if WRITE_SUBMISSION:
    (OUT / "submissions").mkdir(exist_ok=True)
    for _name, _P in [("", P_te), ("_coverage", 0.5 * P_te + 0.5 * np.eye(K)[cover(P_te)])]:   # coverage: argmax = assigned class, ranking kept
        pd.DataFrame(_P, index=pd.Index(np.array(cpds)[TE], name="compound"), columns=classes).reset_index().to_csv(OUT / "submissions" / f"submission_candidate{_name}.csv", index=False)
CALLS.sort_values("n_DEG_top_doses")